# ML Options Paper Trading — V4.8 Controlled Feature Research

Research and paper trading only. No broker, no orders, no profitability claim. V4.4–V4.7 notebooks, scripts, source CSVs and reports are untouched; this notebook writes only `V4_8_*` files (plus the V4.7 audit reports it needs as its gate).

**Question:** do four predeclared feature models (B1 lagged settlement return, B2 lagged underlying return, B3 signed-moneyness bucket, B4 logistic combination) beat the training-rate prior (B0) on held-out dates? Selection uses **validation only**; test is exploratory and never used to choose.

## Run order (Colab)
1. **Cell 1** — upload the 5 `OPTIDX_NIFTY_*.csv` files and every `BhavCopy_NSE_FO_0_0_0_YYYYMMDD_F_0000.csv` you have into `/content`.
2. **Cells 2–5** — write the four scripts (unchanged V4.7 audit + dataset scripts, V4.8 pipeline, tests).
3. **Cell 6 — RUN V4.7 AUDIT** (the gate needs fresh audit reports for exactly these files).
4. **Cell 7 — RUN TESTS** (stops the notebook if any test fails).
5. **Cell 8 — RUN V4.8 PIPELINE** (fails closed if the gate fails; downloads `V4_8_Reports.zip`).

Every result here is exploratory unless it survives the Bonferroni-adjusted refit interval on validation; even then it is not evidence of profitability (label uses settle(t), which is unknown at decision time; no bid/ask, fees or slippage).

In [ ]:
from google.colab import files
uploaded = files.upload()   # skip if the files are already in /content
print(list(uploaded))

In [ ]:
%%writefile /content/v4_7_source_audit.py
"""ML Options Paper Trading V4.7 - automated source-data audit (paper-research only).

Compares the five original NSE NIFTY option CSVs with every official NSE F&O bhavcopy
found in DATA_DIR. Read-only on inputs; writes V4_7_Audit_*.csv reports to OUT_DIR.
Official settlement agreement is NOT evidence of executable prices, liquidity or profit.

Extreme-movement definitions (kept separate, both reported in `selection_definition`):
  threshold_only          |settlement change| > EXTREME_THRESHOLD (default 100%) between consecutive source observations.
  original_selection_only listed in ORIGINAL_SELECTION_FILE (the earlier hand-built list), if that file is present locally.
                          Matched EXACTLY on option_type, strike, prev_date and date, plus expiry / symbol / source_file
                          whenever the file has those columns. Strike/date alone never qualifies; a file without both
                          dates is rejected with an explicit note instead of being guessed at.
  both                    in both definitions.
The original list is only read, never reconstructed; if the file is absent the pipeline says so in the coverage report.

Usage:  python v4_7_source_audit.py [DATA_DIR] [OUT_DIR]      |   python v4_7_source_audit.py --selftest
"""
import glob
import hashlib
import os
import re
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

EXTREME_THRESHOLD = 1.0      # |settlement change| > 100% between consecutive source observations
ORIGINAL_SELECTION_FILE = "V4_7_Extreme_Movement_Official_Verification.csv"   # optional, read-only
PRICE_TOL = 0.005            # prices are quoted to 0.01; anything >= 0.005 apart differs
OUT_PREFIX = "V4_7_Audit_"
SOURCE_GLOB = "OPTIDX_NIFTY_*.csv"
NSE_URL = "https://nsearchives.nseindia.com/content/fo/BhavCopy_NSE_FO_0_0_0_{d}_F_0000.csv.zip"

ST_BOTH = "Official settlement and close matched"
ST_CLOSE_DIFF = "Settlement matched but close differed"
ST_CLOSE_NA = "Settlement matched; source close missing (close not compared)"
ST_SETTLE_DIFF = "Settlement discrepancy"
ST_NO_RECORD = "No official record found"
ST_NO_DATE = "Official date not available locally"
ST_AMBIG = "Ambiguous or unmatched contract"

SRC_REQUIRED = ["Symbol", "Date", "Expiry", "Option type", "Strike Price", "Close", "Settle Price"]
SRC_OPTIONAL = {"Open": "src_open", "High": "src_high", "Low": "src_low", "LTP": "src_ltp",
                "No. of contracts": "src_contracts", "Open Int": "src_open_interest",
                "Underlying Value": "src_underlying"}
UDIFF_REQUIRED = ["TradDt", "FinInstrmTp", "TckrSymb", "XpryDt", "StrkPric", "OptnTp", "ClsPric",
                  "SttlmPric", "TtlTradgVol", "TtlNbOfTxsExctd", "OpnIntrst"]
LEGACY_REQUIRED = ["INSTRUMENT", "SYMBOL", "EXPIRY_DT", "STRIKE_PR", "OPTION_TYP", "CLOSE",
                   "SETTLE_PR", "CONTRACTS", "OPEN_INT", "TIMESTAMP"]
DATE_FORMATS = ["%Y-%m-%d", "%d-%b-%Y", "%d-%b-%y", "%d-%m-%Y", "%d/%m/%Y"]


# ----------------------------------------------------------------------------- helpers
def parse_dates(s):
    """Strict multi-format date parse; returns (Timestamp series, number of unparseable non-null values)."""
    s = s.astype("string").str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for fmt in DATE_FORMATS:
        todo = out.isna() & s.notna()
        if not todo.any():
            break
        out[todo] = pd.to_datetime(s[todo], format=fmt, errors="coerce")
    return out, int((out.isna() & s.notna() & (s != "") & (s != "-")).sum())


def to_num(s):
    return pd.to_numeric(s.astype("string").str.strip().replace({"-": pd.NA, "": pd.NA}), errors="coerce").astype("float64")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def write_csv(df, path):
    """Atomic overwrite so a failed or repeated run never leaves a half-written report."""
    tmp = str(path) + ".tmp"
    df.to_csv(tmp, index=False, encoding="utf-8", lineterminator="\n")
    os.replace(tmp, path)


class Checks:
    def __init__(self):
        self.rows = []

    def add(self, scope, check, count, detail=""):
        self.rows.append({"scope": scope, "check": check, "issues": int(count),
                          "result": "PASS" if count == 0 else "REVIEW", "detail": detail})

    def frame(self):
        return pd.DataFrame(self.rows)


# ----------------------------------------------------------------------------- source
def load_source(data_dir, checks):
    paths = sorted(p for p in Path(data_dir).glob(SOURCE_GLOB))
    # browser duplicates such as "name (1).csv": keep the base name when both exist
    base = {re.sub(r"\s*\(\d+\)(?=\.csv$)", "", p.name) for p in paths}
    keep = []
    for b in sorted(base):
        cands = [p for p in paths if re.sub(r"\s*\(\d+\)(?=\.csv$)", "", p.name) == b]
        keep.append(next((p for p in cands if p.name == b), cands[0]))
    if not keep:
        raise FileNotFoundError(f"No {SOURCE_GLOB} files in {data_dir}")
    frames = []
    for p in keep:
        d = pd.read_csv(p, dtype=str, encoding="utf-8")
        d.columns = [str(c).strip() for c in d.columns]
        missing = [c for c in SRC_REQUIRED if c not in d.columns]
        checks.add(f"source:{p.name}", "required columns present", len(missing), ",".join(missing))
        if missing:
            continue
        d["source_file"] = p.name
        frames.append(d)
    if not frames:
        raise ValueError("No source file has the required columns.")
    r = pd.concat(frames, ignore_index=True)
    r["date"], bad_d = parse_dates(r["Date"])
    r["expiry"], bad_e = parse_dates(r["Expiry"])
    checks.add("source", "unparseable trade dates", bad_d)
    checks.add("source", "unparseable expiry dates", bad_e)
    r["symbol"] = r["Symbol"].str.strip().str.upper()
    r["option_type"] = r["Option type"].str.strip().str.upper()
    r["strike"] = to_num(r["Strike Price"]).round(2)
    r["src_close"] = to_num(r["Close"])
    r["src_settle"] = to_num(r["Settle Price"])
    for col, name in SRC_OPTIONAL.items():
        r[name] = to_num(r[col]) if col in r.columns else np.nan
    checks.add("source", "non-numeric/missing settlement", int(r["src_settle"].isna().sum()))
    checks.add("source", "option type not CE/PE", int((~r["option_type"].isin(["CE", "PE"])).sum()))
    key = ["date", "symbol", "expiry", "option_type", "strike"]
    checks.add("source", "duplicate contract/date keys", int(r.duplicated(key, keep=False).sum()))
    r = r.dropna(subset=["date", "expiry", "strike"]).copy()
    r = r.sort_values(key + ["source_file"]).reset_index(drop=True)
    return r, keep


def add_movements(r):
    k = ["symbol", "expiry", "option_type", "strike"]
    g = r.sort_values(k + ["date"]).groupby(k, sort=False)
    r = g.obj.copy()
    r["prev_date"] = g["date"].shift(1)
    r["prev_src_settle"] = g["src_settle"].shift(1)
    r["src_return"] = r["src_settle"] / r["prev_src_settle"] - 1
    r.loc[~np.isfinite(r["src_return"]), "src_return"] = np.nan
    return r.sort_values(["date", "option_type", "strike"]).reset_index(drop=True)


# ----------------------------------------------------------------------------- official
def sniff(path):
    try:
        cols = [c.strip() for c in pd.read_csv(path, nrows=0, encoding="utf-8").columns]
    except Exception as e:  # unreadable / binary / empty
        return "unreadable", [], str(e)
    if {"TradDt", "FinInstrmTp", "TckrSymb"} <= set(cols):
        return "udiff", cols, ""
    if {"INSTRUMENT", "SYMBOL", "SETTLE_PR", "TIMESTAMP"} <= set(cols):
        return "legacy", cols, ""
    return "other", cols, ""


def load_official(data_dir, checks):
    """Return (normalized NIFTY index-option rows, per-file inventory). Only 'BhavCopy*'/'fo*bhav*' names are tried."""
    cands = sorted({*glob.glob(os.path.join(data_dir, "*.csv")), *glob.glob(os.path.join(data_dir, "*.csv.zip"))})
    cands = [c for c in cands if not Path(c).name.startswith(("V4_", OUT_PREFIX, "OPTIDX_NIFTY_"))]
    inv, frames = [], []
    for c in cands:
        p = Path(c)
        kind, cols, err = sniff(p)
        row = {"file": p.name, "format": kind, "sha256": sha256(p), "nifty_option_rows": 0,
               "trade_dates": "", "note": err}
        if kind not in ("udiff", "legacy"):
            row["note"] = row["note"] or "not an F&O bhavcopy (ignored)"
            inv.append(row)
            continue
        req = UDIFF_REQUIRED if kind == "udiff" else LEGACY_REQUIRED
        missing = [x for x in req if x not in cols]
        checks.add(f"official:{p.name}", "required columns present", len(missing), ",".join(missing))
        if missing:
            row["note"] = "missing columns: " + ",".join(missing)
            inv.append(row)
            continue
        d = pd.read_csv(p, dtype=str, usecols=req)
        if kind == "udiff":
            d = d[(d["TckrSymb"].str.strip() == "NIFTY") & (d["FinInstrmTp"].str.strip() == "IDO")]
            n = pd.DataFrame({"date": parse_dates(d["TradDt"])[0], "symbol": "NIFTY",
                              "expiry": parse_dates(d["XpryDt"])[0], "option_type": d["OptnTp"].str.strip().str.upper(),
                              "strike": to_num(d["StrkPric"]).round(2), "off_close": to_num(d["ClsPric"]),
                              "off_settle": to_num(d["SttlmPric"]), "off_volume": to_num(d["TtlTradgVol"]),
                              "off_transactions": to_num(d["TtlNbOfTxsExctd"]), "off_open_interest": to_num(d["OpnIntrst"])})
            bad = int(parse_dates(d["TradDt"])[1] + parse_dates(d["XpryDt"])[1])
        else:
            d = d[(d["INSTRUMENT"].str.strip() == "OPTIDX") & (d["SYMBOL"].str.strip() == "NIFTY")]
            n = pd.DataFrame({"date": parse_dates(d["TIMESTAMP"])[0], "symbol": "NIFTY",
                              "expiry": parse_dates(d["EXPIRY_DT"])[0], "option_type": d["OPTION_TYP"].str.strip().str.upper(),
                              "strike": to_num(d["STRIKE_PR"]).round(2), "off_close": to_num(d["CLOSE"]),
                              "off_settle": to_num(d["SETTLE_PR"]), "off_volume": to_num(d["CONTRACTS"]),
                              "off_transactions": np.nan, "off_open_interest": to_num(d["OPEN_INT"])})
            bad = int(parse_dates(d["TIMESTAMP"])[1] + parse_dates(d["EXPIRY_DT"])[1])
        checks.add(f"official:{p.name}", "unparseable dates", bad)
        n["official_file"] = p.name
        m = re.search(r"(\d{8})", p.name)
        if m and len(n):
            fdate = pd.to_datetime(m.group(1), format="%Y%m%d", errors="coerce")
            checks.add(f"official:{p.name}", "trade date differs from filename date",
                       int((n["date"].dropna() != fdate).sum()) if pd.notna(fdate) else 0)
        row["nifty_option_rows"] = len(n)
        row["trade_dates"] = "|".join(sorted(n["date"].dropna().dt.strftime("%Y-%m-%d").unique()))
        if len(n) == 0:
            row["note"] = "no NIFTY index-option rows"
        inv.append(row)
        frames.append(n)
    off = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(
        columns=["date", "symbol", "expiry", "option_type", "strike", "off_close", "off_settle", "off_volume",
                 "off_transactions", "off_open_interest", "official_file"])
    key = ["date", "symbol", "expiry", "option_type", "strike"]
    full_dups = off.duplicated(keep="first")
    checks.add("official", "exact duplicate rows (collapsed)", int(full_dups.sum()))
    off = off[~full_dups]
    checks.add("official", "conflicting duplicate contract keys (ambiguous)", int(off.duplicated(key, keep=False).sum()))
    return off, pd.DataFrame(inv)


# ----------------------------------------------------------------------------- compare
EV_UNKNOWN = "Unknown / official date unavailable"
EV_POS = "Positive volume and transactions"
EV_NONE = "No positive volume-and-transaction evidence"
NEW_EXT_COLS = ["liquidity_evidence_class", "prev_official_activity_evidence", "current_official_activity_evidence",
                "in_threshold_definition", "in_original_selection", "selection_definition"]


def liquidity_class(prev_ev, cur_ev):
    pu, cu = prev_ev.startswith("Unknown"), cur_ev.startswith("Unknown")
    if pu and cu:
        return "Liquidity unknown on both dates"
    if cu:
        return "Previous-date activity known; current date unavailable"
    if pu:
        return "Current-date activity known; previous date unavailable"
    pp, cp = prev_ev.startswith("Positive"), cur_ev.startswith("Positive")
    if pp and cp:
        return "Positive volume and transactions on both dates"
    if not pp and not cp:
        return "No positive volume-and-transaction evidence on either date"
    return "Positive volume on " + ("previous" if pp else "current") + " date only"


def compare(src, off):
    key = ["date", "symbol", "expiry", "option_type", "strike"]
    cnt = off.groupby(key).size().rename("official_matches").reset_index()
    first = off.drop_duplicates(key, keep="first")
    m = src.merge(cnt, on=key, how="left").merge(first, on=key, how="left")
    if len(m) != len(src):  # a merge fanned out: refuse rather than double-count a contract-date
        raise ValueError(f"Duplicate contract-date matches: {len(src)} source rows became {len(m)}")
    m["source_duplicate_key"] = src.duplicated(key, keep=False).to_numpy()
    have_dates = set(off["date"].dropna().unique())
    m["official_date_available"] = m["date"].isin(have_dates)
    m["official_matches"] = m["official_matches"].fillna(0).astype(int)
    # contract with same date/type/strike but another expiry exists -> flag for diagnosis
    loose = off.groupby(["date", "option_type", "strike"]).size().rename("n_any_expiry").reset_index()
    m = m.merge(loose, on=["date", "option_type", "strike"], how="left")
    if len(m) != len(src):
        raise ValueError("Duplicate matches introduced by the expiry-diagnosis merge")
    m["n_any_expiry"] = m["n_any_expiry"].fillna(0).astype(int)

    sd = (m["src_settle"] - m["off_settle"]).abs()
    cd = (m["src_close"] - m["off_close"]).abs()
    settle_ok = (sd < PRICE_TOL).fillna(False).to_numpy(dtype=bool)
    close_ok = (cd < PRICE_TOL).fillna(False).to_numpy(dtype=bool)
    status = np.select(
        [m["source_duplicate_key"].to_numpy(dtype=bool), ~m["official_date_available"].to_numpy(dtype=bool),
         (m["official_matches"] > 1).to_numpy(), (m["official_matches"] == 0).to_numpy(), ~settle_ok,
         settle_ok & m["src_close"].isna().to_numpy(), settle_ok & close_ok],
        [ST_AMBIG, ST_NO_DATE, ST_AMBIG, ST_NO_RECORD, ST_SETTLE_DIFF, ST_CLOSE_NA, ST_BOTH],
        default=ST_CLOSE_DIFF)
    m["audit_status"] = status
    unmatched = m["audit_status"].isin([ST_NO_DATE, ST_NO_RECORD, ST_AMBIG])
    m.loc[unmatched, [c for c in off.columns if c.startswith("off_")]] = np.nan  # no single official record to report
    m["settle_diff"] = (m["src_settle"] - m["off_settle"]).round(4)
    m["close_diff"] = (m["src_close"] - m["off_close"]).round(4)
    m["volume_check"] = np.select(
        [m["off_volume"].isna() | m["src_contracts"].isna()], ["not comparable"],
        default=np.where((m["off_volume"] - m["src_contracts"]).abs() < 0.5, "match", "differs"))
    m["oi_check"] = np.select(
        [m["off_open_interest"].isna() | m["src_open_interest"].isna()], ["not comparable"],
        default=np.where((m["off_open_interest"] - m["src_open_interest"]).abs() < 0.5, "match", "differs"))
    # official evidence only: what NSE recorded that day, not proof anything was executable at that price
    m["official_trading_activity"] = np.select(
        [m["off_volume"].isna(), m["off_volume"] > 0], ["unknown", "trades recorded (volume>0)"],
        default="no trades recorded (zero volume; price carried/derived)")
    # liquidity evidence = what NSE recorded (volume, transactions); it does not show a fill was available
    pos = (m["off_volume"] > 0).to_numpy()
    tx_pos = (m["off_transactions"] > 0).to_numpy()
    m["activity_evidence"] = np.select(
        [m["audit_status"].eq(ST_NO_DATE).to_numpy(), m["audit_status"].eq(ST_NO_RECORD).to_numpy(),
         m["audit_status"].eq(ST_AMBIG).to_numpy(), pos & tx_pos, pos & m["off_transactions"].isna().to_numpy()],
        [EV_UNKNOWN, "Unknown / no official record", "Unknown / ambiguous official record",
         EV_POS, "Positive volume (transactions not reported)"],
        default=EV_NONE)
    m["wrong_expiry_only_hint"] = (m["audit_status"] == ST_NO_RECORD) & (m["n_any_expiry"] > 0)
    return m


ORIG_ALIASES = {"option_type": ("option_type", "type"), "strike": ("strike", "strike_price"),
                "prev_date": ("prev_date", "previous_date"), "date": ("date", "current_date"),
                "expiry": ("expiry", "expiry_date"), "symbol": ("symbol",), "source_file": ("source_file",)}
ORIG_REQUIRED = ("option_type", "strike", "prev_date", "date")


def load_original_selection(data_dir):
    """Optional earlier hand-built list. Returns (DataFrame or None, note). Never invents entries."""
    p = Path(data_dir) / ORIGINAL_SELECTION_FILE
    if not p.exists():
        return None, f"{ORIGINAL_SELECTION_FILE} not found locally; original selection could not be applied or reconstructed"
    d = pd.read_csv(p, dtype=str, encoding="utf-8-sig")
    d.columns = [c.strip().lower().replace(" ", "_") for c in d.columns]
    pick = {k: next((c for c in al if c in d.columns), None) for k, al in ORIG_ALIASES.items()}
    missing = [k for k in ORIG_REQUIRED if pick[k] is None]
    if missing:
        return None, (f"{ORIGINAL_SELECTION_FILE} present but cannot identify a movement exactly "
                      f"(missing: {', '.join(missing)}); not used, nothing labelled original")
    out = pd.DataFrame({k: d[c] for k, c in pick.items() if c is not None})
    out["option_type"] = out["option_type"].str.strip().str.upper()
    out["strike"] = to_num(out["strike"]).round(2)
    for c in ("prev_date", "date", "expiry"):
        if c in out:
            out[c] = parse_dates(out[c])[0]
    for c in ("symbol", "source_file"):
        if c in out:
            out[c] = out[c].str.strip()
    out["symbol"] = out["symbol"].str.upper() if "symbol" in out else None
    out = out.drop(columns=[c for c in out.columns if out[c].isna().all() and c not in ORIG_REQUIRED])
    fields = [c for c in out.columns]
    return out.drop_duplicates().reset_index(drop=True), (
        f"{len(d)} rows read from {ORIGINAL_SELECTION_FILE}; exact match on: {', '.join(fields)}")


def match_original(m, orig):
    """Exact identity match of the original list against source movements.
    Returns (mask over m, per-original-row match counts). Zero matches = unmatched, >1 = ambiguous."""
    if orig is None:
        return np.zeros(len(m), dtype=bool), None
    fields = list(orig.columns)
    mm = m[fields].copy()
    mm["_i"] = np.arange(len(m))
    j = orig.reset_index(names="_o").merge(mm, on=fields, how="left")
    counts = j.groupby("_o")["_i"].count()
    mask = np.zeros(len(m), dtype=bool)
    mask[j["_i"].dropna().astype(int).to_numpy()] = True
    return mask, counts


def extreme_report(m, threshold, org=None):
    thr = (m["src_return"].abs() > threshold).to_numpy()
    org = np.zeros(len(m), dtype=bool) if org is None else org & m["prev_date"].notna().to_numpy()
    ext = m[thr | org].copy()
    ext["in_threshold_definition"] = thr[thr | org]
    ext["in_original_selection"] = org[thr | org]
    lookup = m.set_index(["date", "option_type", "strike", "expiry"])
    rows = []
    for i, e in enumerate(ext.sort_values(["date", "option_type", "strike"]).itertuples(), 1):
        rec = {"movement_id": i, "option_type": e.option_type, "strike": e.strike, "expiry": e.expiry.date(),
               "prev_date": e.prev_date.date(), "date": e.date.date(), "gap_calendar_days": (e.date - e.prev_date).days,
               "src_prev_settle": e.prev_src_settle, "src_settle": e.src_settle,
               "src_return_pct": round(e.src_return * 100, 2)}
        stat, ev = [], {}
        for tag, d in (("prev", e.prev_date), ("cur", e.date)):
            k = (d, e.option_type, e.strike, e.expiry)
            row = lookup.loc[k] if k in lookup.index else None
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]
            if row is None:
                st, o = ST_NO_DATE, {}
            else:
                st, o = row["audit_status"], row
            stat.append(st)
            rec[f"{tag}_status"] = st
            for f_, c_ in (("off_settle", "off_settle"), ("off_close", "off_close"), ("off_volume", "off_volume"),
                           ("off_transactions", "off_transactions"), ("off_open_interest", "off_open_interest")):
                rec[f"{tag}_{f_}"] = o[c_] if len(o) else np.nan
            rec[f"{tag}_trading_activity"] = o["official_trading_activity"] if len(o) else "unknown"
            ev[tag] = o["activity_evidence"] if len(o) else EV_UNKNOWN
        rec["official_return_pct"] = (round((rec["cur_off_settle"] / rec["prev_off_settle"] - 1) * 100, 2)
                                      if pd.notna(rec["cur_off_settle"]) and pd.notna(rec["prev_off_settle"])
                                      and rec["prev_off_settle"] else np.nan)
        good = {ST_BOTH, ST_CLOSE_DIFF, ST_CLOSE_NA}
        if any(s == ST_SETTLE_DIFF for s in stat):
            rec["movement_verification"] = "Settlement discrepancy on at least one date"
        elif any(s in (ST_NO_RECORD, ST_AMBIG) for s in stat):
            rec["movement_verification"] = "No/ambiguous official record on an available date"
        elif all(s in good for s in stat):
            rec["movement_verification"] = "Both dates: official settlement matched"
        elif any(s in good for s in stat):
            rec["movement_verification"] = "One date verified; other official date not available locally"
        else:
            rec["movement_verification"] = "Neither official date available locally"
        rec["liquidity_evidence_class"] = liquidity_class(ev["prev"], ev["cur"])
        rec["prev_official_activity_evidence"] = ev["prev"]
        rec["current_official_activity_evidence"] = ev["cur"]
        rec["in_threshold_definition"] = bool(e.in_threshold_definition)
        rec["in_original_selection"] = bool(e.in_original_selection)
        rec["selection_definition"] = ("both" if e.in_threshold_definition and e.in_original_selection else
                                       "threshold_only" if e.in_threshold_definition else "original_selection_only")
        rows.append(rec)
    return pd.DataFrame(rows)


def remaining_dates(m, ext):
    """Dates with no local official file, ranked by unresolved extreme-movement legs, then max |return|, then rows."""
    unresolved = m[m["audit_status"] == ST_NO_DATE]
    legs = []
    if len(ext):
        for tag in ("prev", "cur"):
            t = ext[ext[f"{tag}_status"] == ST_NO_DATE][["date" if tag == "cur" else "prev_date", "src_return_pct"]]
            t.columns = ["date", "ret"]
            legs.append(t)
    legs = pd.concat(legs) if legs else pd.DataFrame(columns=["date", "ret"])
    legs["date"] = pd.to_datetime(legs["date"])
    legs["ret"] = legs["ret"].abs() / 100
    a = unresolved.groupby("date").agg(unresolved_rows=("audit_status", "size"),
                                       max_abs_return_any_row=("src_return", lambda s: s.abs().max())).reset_index()
    b = legs.groupby("date").agg(unresolved_extreme_legs=("ret", "size"), max_abs_return_extreme=("ret", "max")).reset_index()
    out = a.merge(b, on="date", how="left")
    out[["unresolved_extreme_legs"]] = out[["unresolved_extreme_legs"]].fillna(0).astype(int)
    out["max_abs_return_extreme"] = out["max_abs_return_extreme"].fillna(0)
    out["max_abs_return_any_row"] = out["max_abs_return_any_row"].fillna(0)
    out = out.sort_values(["unresolved_extreme_legs", "max_abs_return_extreme", "max_abs_return_any_row",
                           "unresolved_rows", "date"], ascending=[False, False, False, False, True]).reset_index(drop=True)
    out.insert(0, "priority_rank", np.arange(1, len(out) + 1))
    ds = out["date"].dt.strftime("%Y%m%d")
    out["expected_filename"] = "BhavCopy_NSE_FO_0_0_0_" + ds + "_F_0000.csv"
    out["candidate_url_unverified"] = ds.map(lambda d: NSE_URL.format(d=d))
    out["date"] = out["date"].dt.strftime("%Y-%m-%d")
    return out


def coverage(m, ext, rem, inv, threshold, src_files, orig_note=""):
    rows = []
    add = lambda s, k, v: rows.append({"section": s, "metric": k, "value": v})
    add("inputs", "source files", len(src_files))
    add("inputs", "source rows", len(m))
    add("inputs", "source trading dates", m["date"].nunique())
    ok = inv[inv["format"].isin(["udiff", "legacy"]) & (inv["nifty_option_rows"] > 0)] if len(inv) else inv
    add("inputs", "official bhavcopy files processed", len(ok))
    add("inputs", "official files ignored", len(inv) - len(ok))
    add("inputs", "official dates available (all)", int(m.loc[m["official_date_available"], "date"].nunique()))
    for s, n in m["audit_status"].value_counts().items():
        add("row_status", s, int(n))
    for (ot, s), n in m.groupby(["option_type", "audit_status"]).size().items():
        add(f"row_status_{ot}", s, int(n))
    cov = m["official_date_available"].mean()
    add("coverage", "share of source rows whose official date is available locally", round(float(cov), 4))
    add("coverage", "source dates still missing locally", int(rem.shape[0]))
    sm = m[m["audit_status"].isin([ST_BOTH, ST_CLOSE_DIFF, ST_CLOSE_NA, ST_SETTLE_DIFF])]
    add("coverage", "rows with an official match", len(sm))
    add("coverage", "wrong-expiry-only hints (same date/type/strike, other expiry)", int(m["wrong_expiry_only_hint"].sum()))
    add("extreme", f"movements with |source change| > {threshold:.0%}", len(ext))
    if len(ext):
        for s, n in ext["movement_verification"].value_counts().items():
            add("extreme_status", s, int(n))
    if len(ext):
        for s_, n in ext["liquidity_evidence_class"].value_counts().items():
            add("liquidity_class", s_, int(n))
        for s_, n in ext["selection_definition"].value_counts().items():
            add("selection_definition", s_, int(n))
    add("selection_definition", "original selection source", orig_note)
    add("selection_definition", "threshold definition",
        f"|settlement change| > {threshold:.0%} between consecutive source observations")
    for th in (0.5, 0.75, 0.9, 1.0):
        add("extreme_threshold_counts", f"|change| > {th:.0%}", int((m["src_return"].abs() > th).sum()))
    add("note", "scope", "Official settlement agreement verifies the recorded price only; it is not evidence of "
                          "executable prices, liquidity, fills or profitability.")
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------- orchestration
def run_audit(data_dir="/content", out_dir=None, threshold=EXTREME_THRESHOLD, verbose=True):
    data_dir = str(data_dir)
    out_dir = Path(out_dir or data_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    checks = Checks()
    src, src_paths = load_source(data_dir, checks)
    src = add_movements(src)
    off, inv = load_official(data_dir, checks)
    m = compare(src, off)
    checks.add("compare", "ambiguous official matches", int((m["audit_status"] == ST_AMBIG).sum()))
    checks.add("compare", "settlement discrepancies", int((m["audit_status"] == ST_SETTLE_DIFF).sum()),
               "discrepancy = source differs from official; no value has been altered")
    checks.add("compare", "source contract-date keys duplicated (marked ambiguous)", int(m["source_duplicate_key"].sum()))
    checks.add("compare", "same date/type/strike under more than one source expiry",
               int(src.groupby(["date", "option_type", "strike"])["expiry"].nunique().gt(1).sum()))
    checks.add("compare", "contract not found but another expiry exists officially (identity check)",
               int(m["wrong_expiry_only_hint"].sum()))
    orig, orig_note = load_original_selection(data_dir)
    org_mask, org_counts = match_original(m, orig)
    ext = extreme_report(m, threshold, org_mask)
    checks.add("extreme", "movement ids not unique", int(ext["movement_id"].duplicated().sum()) if len(ext) else 0)
    if orig is None and "cannot identify" in orig_note:
        checks.add("extreme", "original-selection file unusable (insufficient identity columns)", 1, orig_note)
    if org_counts is not None:
        checks.add("extreme", "original-selection rows with no exact movement match", int((org_counts == 0).sum()))
        checks.add("extreme", "original-selection rows matching more than one movement (ambiguous)",
                   int((org_counts > 1).sum()))
    rem = remaining_dates(m, ext)
    cov = coverage(m, ext, rem, inv, threshold, src_paths, orig_note)

    keep = ["source_file", "date", "symbol", "expiry", "option_type", "strike", "audit_status",
            "src_settle", "off_settle", "settle_diff", "src_close", "off_close", "close_diff",
            "src_contracts", "off_volume", "volume_check", "off_transactions",
            "src_open_interest", "off_open_interest", "oi_check", "official_trading_activity",
            "official_file", "official_matches", "wrong_expiry_only_hint", "prev_date", "src_return"]
    rl = m[keep].copy()
    for c in ("date", "expiry", "prev_date"):
        rl[c] = rl[c].dt.strftime("%Y-%m-%d")
    write_csv(rl, out_dir / f"{OUT_PREFIX}Row_Level_Comparison.csv")
    write_csv(ext.drop(columns=NEW_EXT_COLS, errors="ignore"), out_dir / f"{OUT_PREFIX}Extreme_Movement_Verification.csv")
    write_csv(ext, out_dir / f"{OUT_PREFIX}Extreme_Movement_Liquidity_Review.csv")  # consolidated report
    if len(ext):
        summ = (ext.groupby(["movement_verification", "liquidity_evidence_class"]).size().rename("movement_count")
                .reset_index().sort_values(["movement_verification", "movement_count", "liquidity_evidence_class"],
                                           ascending=[True, False, True]))
    else:
        summ = pd.DataFrame(columns=["movement_verification", "liquidity_evidence_class", "movement_count"])
    write_csv(summ, out_dir / f"{OUT_PREFIX}Liquidity_Review_Summary.csv")
    write_csv(rem, out_dir / f"{OUT_PREFIX}Remaining_Dates_Manifest.csv")
    write_csv(cov, out_dir / f"{OUT_PREFIX}Coverage_Summary.csv")
    write_csv(checks.frame(), out_dir / f"{OUT_PREFIX}Validation_Checks.csv")
    manifest = pd.concat([inv.assign(role="official_candidate"),
                          pd.DataFrame({"file": [p.name for p in src_paths], "sha256": [sha256(p) for p in src_paths],
                                        "format": "source", "role": "source"})], ignore_index=True)
    write_csv(manifest, out_dir / f"{OUT_PREFIX}Input_Manifest.csv")

    if verbose:
        print("=" * 72, "\nV4.7 SOURCE AUDIT SUMMARY (paper research; settlement check != executability)\n" + "=" * 72)
        print(f"Source rows: {len(m)} | source dates: {m['date'].nunique()} | official files processed: "
              f"{int(((inv['format'].isin(['udiff','legacy'])) & (inv['nifty_option_rows'] > 0)).sum()) if len(inv) else 0}")
        print(m["audit_status"].value_counts().to_string())
        print(f"\nExtreme movements (> {threshold:.0%}): {len(ext)}")
        if len(ext):
            print(ext["movement_verification"].value_counts().to_string())
            print("\nLiquidity-evidence classes (official volume/transactions only; not proof of executability):")
            print(ext["liquidity_evidence_class"].value_counts().to_string())
            print("Selection definitions:", ext["selection_definition"].value_counts().to_dict(), "|", orig_note)
        print(f"\nOfficial dates still missing: {len(rem)} (top 10 in manifest by priority)")
        print(rem.head(10)[["priority_rank", "date", "unresolved_extreme_legs", "max_abs_return_extreme",
                            "unresolved_rows"]].to_string(index=False))
        bad = checks.frame()
        bad = bad[bad["result"] == "REVIEW"]
        print("\nValidation items needing review:", "none" if bad.empty else "")
        if len(bad):
            print(bad[["scope", "check", "issues"]].to_string(index=False))
        print("\nReports saved to", out_dir)
    return {"row_level": rl, "extreme": ext, "remaining": rem, "coverage": cov, "checks": checks.frame(), "inventory": inv}


# ----------------------------------------------------------------------------- self-test
def run_selftest():
    tmp = Path(tempfile.mkdtemp(prefix="v47test_"))
    src = pd.DataFrame({
        "Symbol  ": ["NIFTY"] * 11, "Date  ": ["10-Mar-2026"] * 7 + ["11-Mar-2026"] * 3 + ["12-Mar-2026"],
        "Expiry  ": ["29-Sep-2026"] * 11, "Option type  ": ["CE"] * 11,
        "Strike Price  ": ["100", "200", "300", "400", "500", "600", "800", "100", "100", "800", "100"],
        "Close  ": ["10", "10", "10", "-", "10", "10", "10", "10", "10", "20", "10"],
        "Settle Price  ": ["5", "5", "5", "5", "5", "5", "5", "5", "5", "20", "5"]})
    src.to_csv(tmp / "OPTIDX_NIFTY_CE_01-Mar-2026_TO_31-Mar-2026.csv", index=False)

    def u(strike, exp, close, settle):
        return {"TradDt": "2026-03-10", "FinInstrmTp": "IDO", "TckrSymb": "NIFTY", "XpryDt": exp, "StrkPric": strike,
                "OptnTp": "CE", "ClsPric": close, "SttlmPric": settle, "TtlTradgVol": 3, "TtlNbOfTxsExctd": 2, "OpnIntrst": 9}
    rows = [u("100.00", "2026-09-29", "10.00", "5.00"),      # both match
            u("200.00", "2026-09-29", "11.00", "5.00"),      # close differs
            u("300.00", "2026-09-29", "10.00", "6.00"),      # settlement discrepancy
            u("400.00", "2026-09-29", "10.00", "5.00"),      # source close missing
            u("600.00", "2026-12-29", "10.00", "5.00"),      # only other expiry -> no record
            u("500.00", "2026-09-29", "10.00", "5.00"), u("500.00", "2026-09-29", "10.00", "7.00"),  # ambiguous
            u("800.00", "2026-09-29", "10.00", "5.00")]                                               # extreme leg 1
    pd.DataFrame(rows).to_csv(tmp / "BhavCopy_NSE_FO_0_0_0_20260310_F_0000.csv", index=False)
    pd.DataFrame({"INSTRUMENT": ["OPTIDX"] * 2, "SYMBOL": ["NIFTY"] * 2, "EXPIRY_DT": ["29-Sep-2026"] * 2,
                  "STRIKE_PR": [100, 800], "OPTION_TYP": ["CE"] * 2, "CLOSE": [10, 20], "SETTLE_PR": [5, 20],
                  "CONTRACTS": [3, 0], "OPEN_INT": [9, 9], "TIMESTAMP": ["11-Mar-2026"] * 2}
                 ).to_csv(tmp / "fo11MAR2026bhav.csv", index=False)  # legacy format; extreme leg 2 has zero contracts
    pd.DataFrame({"option_type": ["CE", "PE", "CE", "CE"], "strike": [800, 9999, 800, 800],
                  "prev_date": ["2026-03-10", "2026-03-10", "2026-03-09", "2026-03-10"], "date": ["2026-03-11"] * 4,
                  "expiry": ["2026-09-29", "2026-09-29", "2026-09-29", "2026-12-29"]}  # exact / absent / decoy prev_date / wrong expiry
                 ).to_csv(tmp / ORIGINAL_SELECTION_FILE, index=False)
    pd.DataFrame({"DATE": ["12-MAR-2026"], "INSTRUMENT": ["OPTIDX"]}).to_csv(tmp / "FOSett_prce_12032026.csv", index=False)
    res = run_audit(tmp, verbose=False)
    got = res["row_level"].set_index(["date", "strike"])["audit_status"].to_dict()
    exp = {("2026-03-10", 100.0): ST_BOTH, ("2026-03-10", 200.0): ST_CLOSE_DIFF, ("2026-03-10", 300.0): ST_SETTLE_DIFF,
           ("2026-03-10", 400.0): ST_CLOSE_NA, ("2026-03-10", 500.0): ST_AMBIG, ("2026-03-10", 600.0): ST_NO_RECORD,
           ("2026-03-10", 800.0): ST_BOTH, ("2026-03-11", 100.0): ST_AMBIG,  # duplicate source key
           ("2026-03-11", 800.0): ST_BOTH, ("2026-03-12", 100.0): ST_NO_DATE}
    assert got == exp, (got, exp)
    ext = res["extreme"]
    assert len(ext) == 1 and ext.iloc[0]["selection_definition"] == "both", ext
    assert ext.iloc[0]["liquidity_evidence_class"] == "Positive volume on previous date only", ext.iloc[0]
    assert res["checks"].set_index("check").loc["source contract-date keys duplicated (marked ambiguous)", "issues"] == 2
    assert res["checks"].set_index("check").loc["original-selection rows with no exact movement match", "issues"] == 3
    # a file lacking prev_date must label nothing and say why
    (tmp / ORIGINAL_SELECTION_FILE).write_text("option_type,strike,date\nCE,800,2026-03-11\n")
    r2 = run_audit(tmp, verbose=False)
    assert r2["extreme"]["selection_definition"].eq("threshold_only").all()
    assert r2["checks"]["check"].str.contains("original-selection file unusable").any()
    (tmp / ORIGINAL_SELECTION_FILE).unlink()
    r3 = run_audit(tmp, verbose=False)
    assert "not found locally" in r3["coverage"].set_index("metric").loc["original selection source", "value"]
    (tmp / ORIGINAL_SELECTION_FILE).write_text("option_type,strike,prev_date,date,expiry\nCE,800,2026-03-10,2026-03-11,2026-09-29\n")
    for a_, b_, want in [(EV_UNKNOWN, EV_UNKNOWN, "Liquidity unknown on both dates"),
                         (EV_POS, EV_UNKNOWN, "Previous-date activity known; current date unavailable"),
                         (EV_UNKNOWN, EV_NONE, "Current-date activity known; previous date unavailable"),
                         (EV_POS, EV_POS, "Positive volume and transactions on both dates"),
                         (EV_NONE, EV_NONE, "No positive volume-and-transaction evidence on either date"),
                         (EV_NONE, EV_POS, "Positive volume on current date only")]:
        assert liquidity_class(a_, b_) == want, (a_, b_)
    rl_ = res["row_level"]
    assert rl_.loc[(rl_["date"] == "2026-03-10") & (rl_["strike"] == 600.0), "wrong_expiry_only_hint"].all()
    assert "FOSett_prce_12032026.csv" in set(res["inventory"].loc[res["inventory"]["format"] == "other", "file"])
    run_audit(tmp, verbose=False)
    h1 = {p.name: sha256(p) for p in tmp.glob(OUT_PREFIX + "*.csv")}
    run_audit(tmp, verbose=False)
    h2 = {p.name: sha256(p) for p in tmp.glob(OUT_PREFIX + "*.csv")}
    assert h1 == h2 and len(h1) == 8, "rerun changed outputs"
    print("SELFTEST PASSED: status cases, duplicate-key ambiguity, liquidity classes, two selection definitions, idempotent rerun")


if __name__ == "__main__":
    if "--selftest" in sys.argv:
        run_selftest()
    else:
        a = [x for x in sys.argv[1:] if not x.startswith("--")]
        run_audit(a[0] if a else "/content", a[1] if len(a) > 1 else None)


In [ ]:
%%writefile /content/v4_7_model_dataset.py
"""ML Options Paper Trading V4.7 - separate modelling dataset + baseline comparison (paper research only).

Runs AFTER v4_7_source_audit.py. Gate: the audit's Validation_Checks must exist with no REVIEW rows (override: force=True).
Read-only on source files; writes V4_7_Model_*.csv. No rows are deleted from the dataset: ineligible and purged rows are kept
with an explicit reason. Nothing here claims predictive skill or profitability.

Label: target_next_up = settle(t+1) > settle(t) for the SAME contract, only when t+1 is the next trading day in the source calendar.
Features use data up to t-1 only (same four features as V4.4-V4.6): settle_return_lag1/lag2, underlying_return_lag1, moneyness_lag1.
Splits: chronological by calendar date, TRAIN_FRAC/VAL_FRAC/rest. Purge: a row is dropped from its split when its target date
(t+1) falls on or after the next split's first date, so no training label is decided by a later split's prices.

Usage: python v4_7_model_dataset.py [DATA_DIR] [OUT_DIR]    |    python v4_7_model_dataset.py --selftest
"""
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))  # -I / Colab: find the audit module next to this file
import v4_7_source_audit as A

MODEL_PREFIX = "V4_7_Model_"
PRICE_FLOOR = 1.0            # rupees; applies to settle at t, t-1, t-2 only (never to t+1, which would select on the outcome)
TRAIN_FRAC, VAL_FRAC = 0.6, 0.2
FEATURES = ["settle_return_lag1", "settle_return_lag2", "underlying_return_lag1", "moneyness_lag1"]
MONEYNESS_BINS = [0, 0.9, 1.1, np.inf]   # fixed in advance, not fitted
SHRINK = 20                  # m-estimate strength toward the training prior for table baselines
BOOT = 2000

RULES = [
    ("R1_features", "All four features exist", "Lags need consecutive trading days in the source calendar; gaps give NaN, never a stale price."),
    ("R2_target", "Next-trading-day settlement exists for the same contract", "Label is a one-day move; a multi-day gap is a different horizon."),
    ("R3_price_floor", f"min(settle t, t-1, t-2) >= {PRICE_FLOOR}", "Tick-size (0.05) noise dominates sub-rupee prices. Uses past/current prices only, so it cannot select on the outcome."),
    ("R4_official_clean", "No official settlement discrepancy / ambiguous identity in the V4.7 audit", "Rows contradicted by NSE or with unclear contract identity are not modelled. Unverified rows are kept and labelled."),
    ("R5_unique_key", "Single row per contract-date", "Duplicate keys make lags ambiguous."),
]


# ----------------------------------------------------------------------------- build
def gate(out_dir, force):
    p = Path(out_dir) / f"{A.OUT_PREFIX}Validation_Checks.csv"
    if not p.exists():
        raise RuntimeError("Run the source audit first: Validation_Checks.csv not found.")
    bad = pd.read_csv(p)
    bad = bad[bad["result"] == "REVIEW"]
    if len(bad) and not force:
        raise RuntimeError(f"Source audit has {len(bad)} unresolved REVIEW checks; fix or pass force=True.\n"
                           + bad[["scope", "check", "issues"]].to_string(index=False))
    return len(bad)


def build_features(r):
    """r: source rows from A.load_source (with src_underlying). Returns dataframe with features, target, calendar index."""
    cal = np.array(sorted(r["date"].unique()))
    cidx = {d: i for i, d in enumerate(cal)}
    r = r.copy()
    r["cal_idx"] = r["date"].map(cidx)
    und = r.groupby("date")["src_underlying"].agg(["first", "nunique"]).reindex(cal)
    ucal = und["first"].to_numpy(dtype=float)
    k = ["symbol", "expiry", "option_type", "strike"]
    r = r.sort_values(k + ["date"]).reset_index(drop=True)
    g = r.groupby(k, sort=False)

    def lag(col, n):  # value n rows back, only if that row is exactly n trading days back
        ok = (r["cal_idx"] - g["cal_idx"].shift(n)) == n
        return g[col].shift(n).where(ok)
    s1, s2, s3 = lag("src_settle", 1), lag("src_settle", 2), lag("src_settle", 3)
    nxt = g["src_settle"].shift(-1).where((g["cal_idx"].shift(-1) - r["cal_idx"]) == 1)
    r["settle_t"], r["settle_t1"], r["settle_t2"] = r["src_settle"], s1, s2
    r["settle_return_lag1"] = s1 / s2 - 1
    r["settle_return_lag2"] = s2 / s3 - 1
    i = r["cal_idx"].to_numpy()
    u1 = np.where(i >= 1, ucal[np.maximum(i - 1, 0)], np.nan)
    u2 = np.where(i >= 2, ucal[np.maximum(i - 2, 0)], np.nan)
    r["underlying_return_lag1"] = u1 / u2 - 1
    r["moneyness_lag1"] = u1 / r["strike"]
    r["settle_next"] = nxt
    r["target_next_up"] = (nxt > r["src_settle"]).astype(float).where(nxt.notna())
    r["target_tie"] = (nxt == r["src_settle"]) & nxt.notna()
    r["target_date"] = pd.Series(cal[np.minimum(i + 1, len(cal) - 1)], index=r.index).where(nxt.notna())
    for c in FEATURES:
        r.loc[~np.isfinite(r[c]), c] = np.nan
    r["extreme_return_flag"] = (r[["settle_return_lag1", "settle_return_lag2"]].abs() > 1).any(axis=1)
    return r, cal, int((und["nunique"] > 1).sum())


def apply_rules(r, rowlevel):
    key = ["date", "option_type", "strike", "expiry"]
    rl = rowlevel[["date", "option_type", "strike", "expiry", "audit_status"]].copy()
    for c in ("date", "expiry"):
        rl[c] = pd.to_datetime(rl[c])
    rl = rl.drop_duplicates(key)
    r = r.merge(rl, on=key, how="left")
    r["official_status"] = r["audit_status"].fillna("not in audit")
    r["R1_features"] = r[FEATURES].notna().all(axis=1)
    r["R2_target"] = r["target_next_up"].notna()
    r["R3_price_floor"] = r[["settle_t", "settle_t1", "settle_t2"]].min(axis=1) >= PRICE_FLOOR
    bad = {A.ST_SETTLE_DIFF, A.ST_AMBIG, "not in audit"}
    r["R4_official_clean"] = ~r["official_status"].isin(bad)
    r["R5_unique_key"] = ~r.duplicated(["symbol", "expiry", "option_type", "strike", "date"], keep=False)
    rules = [x[0] for x in RULES]
    r["eligible"] = r[rules].all(axis=1)
    r["exclusion_reason"] = r[rules].apply(lambda row: "|".join(k for k in rules if not row[k]), axis=1)
    return r


def assign_splits(r, cal):
    n = len(cal)
    val_start, test_start = int(n * TRAIN_FRAC), int(n * (TRAIN_FRAC + VAL_FRAC))
    base = np.where(r["cal_idx"] < val_start, "train", np.where(r["cal_idx"] < test_start, "validation", "test"))
    tgt_idx = r["cal_idx"] + 1
    purged = ((base == "train") & (tgt_idx >= val_start)) | ((base == "validation") & (tgt_idx >= test_start))
    r["split"] = np.where(purged, "purged", base)
    r["split_if_not_purged"] = base
    return r, cal[val_start], cal[test_start]


def leakage_checks(r, val_start, test_start, checks):
    ok = r[r["eligible"] & r["split"].isin(["train", "validation"])]
    checks.add("model", "train rows whose target date >= validation start",
               int(((ok["split"] == "train") & (ok["target_date"] >= val_start)).sum()))
    checks.add("model", "validation rows whose target date >= test start",
               int(((ok["split"] == "validation") & (ok["target_date"] >= test_start)).sum()))
    checks.add("model", "eligible rows with target date not after feature date", int((ok["target_date"] <= ok["date"]).sum()))
    checks.add("model", "duplicate contract-date keys among eligible rows",
               int(r[r["eligible"]].duplicated(["symbol", "expiry", "option_type", "strike", "date"]).sum()))
    checks.add("model", "eligible rows with non-consecutive lag (settle_t2 missing)", int(r.loc[r["eligible"], "settle_t2"].isna().sum()))


# ----------------------------------------------------------------------------- baselines
def auc(y, s):
    y = np.asarray(y)
    n1, n0 = y.sum(), len(y) - y.sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    rk = pd.Series(s).rank().to_numpy()
    return (rk[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def table_prob(train_key, train_y, test_key, prior):
    df = pd.DataFrame({"k": train_key, "y": train_y}).groupby("k")["y"].agg(["sum", "count"])
    p = (df["sum"] + SHRINK * prior) / (df["count"] + SHRINK)
    return pd.Series(test_key).map(p).fillna(prior).to_numpy()


def sign3(x):
    return np.sign(np.asarray(x, dtype=float)).astype(int)


def baseline_probs(tr, te):
    prior = float(tr["target_next_up"].mean())
    y = tr["target_next_up"].to_numpy()
    mn = lambda d: pd.cut(d["moneyness_lag1"], MONEYNESS_BINS, labels=False, right=False).to_numpy()
    return {
        "B0_training_prior": np.full(len(te), np.clip(prior, 0.01, 0.99)),
        "B1_settle_return_lag1_sign": table_prob(sign3(tr["settle_return_lag1"]), y, sign3(te["settle_return_lag1"]), prior),
        "B2_underlying_return_lag1_sign": table_prob(sign3(tr["underlying_return_lag1"]), y, sign3(te["underlying_return_lag1"]), prior),
        "B3_moneyness_bucket": table_prob(mn(tr), y, mn(te), prior),
    }


def baseline_comparison(d):
    rng = np.random.default_rng(0)
    rows = []
    for ot in sorted(d["option_type"].unique()):
        tr = d[(d["split"] == "train") & (d["option_type"] == ot)]
        if tr["target_next_up"].nunique() < 2:
            continue
        for sp in ("validation", "test"):
            te = d[(d["split"] == sp) & (d["option_type"] == ot)].reset_index(drop=True)
            if te.empty:
                rows.append({"option_type": ot, "split": sp, "model": "(no eligible rows)", "rows": 0})
                continue
            y = te["target_next_up"].to_numpy()
            probs = baseline_probs(tr, te)
            dates = te["date"].to_numpy()
            groups = [np.where(dates == u)[0] for u in np.unique(dates)]
            b0 = probs["B0_training_prior"]
            for name, p in probs.items():
                brier = float(np.mean((p - y) ** 2))
                diffs, aucs = [], []
                for _ in range(BOOT):
                    ix = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
                    diffs.append(np.mean((p[ix] - y[ix]) ** 2) - np.mean((b0[ix] - y[ix]) ** 2))
                    aucs.append(auc(y[ix], p[ix]))
                lo, hi = np.percentile(diffs, [2.5, 97.5])
                pc = np.clip(p, 1e-6, 1 - 1e-6)
                rows.append({
                    "option_type": ot, "split": sp, "model": name, "rows": len(te), "dates": len(groups),
                    "train_rows": len(tr), "train_up_rate": round(float(tr["target_next_up"].mean()), 4),
                    "eval_up_rate": round(float(y.mean()), 4), "brier": round(brier, 5),
                    "brier_vs_B0": round(brier - float(np.mean((b0 - y) ** 2)), 5),
                    "brier_diff_ci_lo": round(lo, 5), "brier_diff_ci_hi": round(hi, 5),
                    "log_loss": round(float(-np.mean(y * np.log(pc) + (1 - y) * np.log(1 - pc))), 5),
                    "accuracy_at_0.5": round(float(((p >= 0.5) == (y == 1)).mean()), 4),
                    "auc": round(float(auc(y, p)), 4), "auc_ci_lo": round(float(np.nanpercentile(aucs, 2.5)), 4),
                    "auc_ci_hi": round(float(np.nanpercentile(aucs, 97.5)), 4),
                    "evidence_vs_B0": ("n/a (reference)" if name.startswith("B0") else
                                       "interval excludes 0 (better)" if hi < 0 else
                                       "interval excludes 0 (worse)" if lo > 0 else "no evidence of difference"),
                    "note": "uncorrected for multiple comparisons" + ("; test window overlaps data seen in earlier experiments" if sp == "test" else "")})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------- orchestration
def run_model_dataset(data_dir="/content", out_dir=None, force=False, verbose=True):
    data_dir, out_dir = str(data_dir), Path(out_dir or data_dir)
    n_review = gate(out_dir, force)
    checks = A.Checks()
    src, _ = A.load_source(data_dir, checks)
    r, cal, n_und_conflict = build_features(src)
    checks.add("model", "dates with more than one underlying value", n_und_conflict)
    rowlevel = pd.read_csv(out_dir / f"{A.OUT_PREFIX}Row_Level_Comparison.csv")
    r = apply_rules(r, rowlevel)
    r, val_start, test_start = assign_splits(r, cal)
    leakage_checks(r, val_start, test_start, checks)
    elig = r[r["eligible"] & r["split"].isin(["train", "validation", "test"])]

    cols = ["source_file", "symbol", "expiry", "option_type", "strike", "date", "target_date", "cal_idx", "settle_t",
            "settle_next"] + FEATURES + ["target_next_up", "target_tie", "extreme_return_flag", "official_status",
                                          "eligible", "exclusion_reason", "split", "split_if_not_purged"]
    ds = r[cols].copy()
    for c in ("expiry", "date", "target_date"):
        ds[c] = ds[c].dt.strftime("%Y-%m-%d")
    ds = ds.sort_values(["date", "option_type", "strike"]).reset_index(drop=True)
    write_csv = A.write_csv
    write_csv(ds, out_dir / f"{MODEL_PREFIX}Dataset.csv")
    write_csv(pd.DataFrame(RULES, columns=["rule", "condition", "rationale"]), out_dir / f"{MODEL_PREFIX}Eligibility_Rules.csv")
    rule_ids = [x[0] for x in RULES]
    elig_sum = pd.DataFrame({"rule": rule_ids,
                             "rows_failing": [int((~r[k]).sum()) for k in rule_ids],
                             "rows_failing_first": [int(r["exclusion_reason"].str.split("|").str[0].eq(k).sum()) for k in rule_ids]})
    elig_sum.loc[len(elig_sum)] = ["ELIGIBLE (all rules)", int(r["eligible"].sum()), 0]
    elig_sum.loc[len(elig_sum)] = ["total rows", len(r), 0]
    write_csv(elig_sum, out_dir / f"{MODEL_PREFIX}Eligibility_Summary.csv")
    sp = (r[r["eligible"] | (r["split"] == "purged")].assign(is_eligible=lambda x: x["eligible"])
          .groupby(["split", "option_type"]).agg(rows=("eligible", "size"), eligible_rows=("eligible", "sum"),
                                                 dates=("date", "nunique"), first_date=("date", "min"), last_date=("date", "max"),
                                                 up_rate=("target_next_up", "mean")).reset_index())
    sp["first_date"], sp["last_date"] = sp["first_date"].dt.strftime("%Y-%m-%d"), sp["last_date"].dt.strftime("%Y-%m-%d")
    write_csv(sp.round(4), out_dir / f"{MODEL_PREFIX}Split_Summary.csv")
    cmp_ = baseline_comparison(elig)
    write_csv(cmp_, out_dir / f"{MODEL_PREFIX}Baseline_Comparison.csv")
    write_csv(checks.frame(), out_dir / f"{MODEL_PREFIX}Validation_Checks.csv")

    if verbose:
        print("=" * 72, "\nV4.7 MODELLING DATASET + BASELINES (exploratory; no skill or profit claim)\n" + "=" * 72)
        print(f"Audit gate: passed ({n_review} unresolved review checks)")
        print(f"Calendar dates: {len(cal)} | validation starts {pd.Timestamp(val_start).date()} | test starts {pd.Timestamp(test_start).date()}")
        print(f"Rows: {len(r)} | eligible: {int(r['eligible'].sum())} | purged at boundaries: {int((r['split'] == 'purged').sum())}")
        print(elig_sum.to_string(index=False))
        print(sp.to_string(index=False))
        show = ["option_type", "split", "model", "rows", "dates", "brier", "brier_vs_B0", "brier_diff_ci_lo", "brier_diff_ci_hi",
                "auc", "evidence_vs_B0"]
        print(cmp_[[c for c in show if c in cmp_.columns]].to_string(index=False))
        bad = checks.frame()
        bad = bad[bad["result"] == "REVIEW"]
        print("\nModel validation items needing review:", "none" if bad.empty else "")
        if len(bad):
            print(bad[["scope", "check", "issues"]].to_string(index=False))
        print("\nReports saved to", out_dir)
    return {"dataset": ds, "comparison": cmp_, "checks": checks.frame(), "split_summary": sp, "eligibility": elig_sum}


# ----------------------------------------------------------------------------- self-test
def run_selftest():
    tmp = Path(tempfile.mkdtemp(prefix="v47model_"))
    rng = np.random.default_rng(1)
    dates = pd.bdate_range("2026-01-05", periods=30)
    rows = []
    for ot in ("CE", "PE"):
        for strike in (100, 200):
            px = 50.0
            for i, d in enumerate(dates):
                px = max(px * (1 + rng.normal(0, 0.05)), 2.0)
                if ot == "CE" and strike == 100 and i == 10:   # missing observation -> lag/target gaps
                    continue
                rows.append({"Symbol": "NIFTY", "Date": d.strftime("%d-%b-%Y"), "Expiry": "29-Sep-2026", "Option type": ot,
                             "Strike Price": strike, "Close": round(px, 2), "Settle Price": round(px, 2),
                             "Underlying Value": 20000 + 10 * i})
    pd.DataFrame(rows).to_csv(tmp / "OPTIDX_NIFTY_CE_01-Jan-2026_TO_31-Mar-2026.csv", index=False)
    A.run_audit(tmp, verbose=False)
    res = run_model_dataset(tmp, verbose=False)
    ds = res["dataset"]
    assert (res["checks"]["issues"] == 0).all(), res["checks"]
    c = ds[(ds.option_type == "CE") & (ds.strike == 100)].set_index("date")
    d10, d9, d11, d12 = (dates[i].strftime("%Y-%m-%d") for i in (10, 9, 11, 12))
    assert pd.isna(c.loc[d9, "target_next_up"]), "row before a gap must have no target"
    assert pd.isna(c.loc[d11, "settle_return_lag1"]) and pd.isna(c.loc[d12, "settle_return_lag1"]), "lags must not span a gap"
    assert set(ds.loc[ds.split == "purged", "split_if_not_purged"]) == {"train", "validation"}
    e = ds[ds.eligible & ds.split.isin(["train", "validation"])]
    vs = ds.loc[ds.split == "validation", "date"].min()
    assert (e.loc[e.split == "train", "target_date"] < vs).all()
    h1 = {p.name: A.sha256(p) for p in tmp.glob(MODEL_PREFIX + "*.csv")}
    run_model_dataset(tmp, verbose=False)
    assert h1 == {p.name: A.sha256(p) for p in tmp.glob(MODEL_PREFIX + "*.csv")} and len(h1) == 6, "rerun changed outputs"
    print("MODEL SELFTEST PASSED: gap handling, purge, no cross-split labels, eligibility rules, idempotent rerun")


if __name__ == "__main__":
    if "--selftest" in sys.argv:
        run_selftest()
    else:
        a = [x for x in sys.argv[1:] if not x.startswith("--")]
        run_model_dataset(a[0] if a else "/content", a[1] if len(a) > 1 else None)


In [ ]:
%%writefile /content/v4_8_feature_research.py
"""ML Options Paper Trading V4.8 - controlled feature research (paper research only; no broker, no orders).

Question: do a few PREDECLARED features add out-of-sample value over a training-rate prior?
Reuses the V4.7 scripts unchanged (audit, lag/target/eligibility/split code). Defects found in V4.7 are fixed here only:
  D1 moneyness buckets were option-type-blind (same S/K bin = ITM for calls, OTM for puts).      -> signed log-moneyness, ITM>0 for both.
  D2 sign-table baselines mapped NaN to an arbitrary integer category.                           -> explicit MISSING/unseen policy.
  D3 audit gate only looked for REVIEW rows; it never checked the reports belong to the current inputs. -> hash + row-count gate (fail closed).
  D4 confidence intervals resampled evaluation dates only (fitted parameters treated as fixed).   -> adds train+eval date resampling WITH refit.
  D5 no multiplicity handling.                                                                    -> Bonferroni over the 8 predeclared validation comparisons.

PREDECLARED EXPERIMENT (fixed before any V4.8 comparison was run)
  Decision time: start of trading day t. Everything used is an end-of-day value from day t-1 or earlier. The label compares
  settle(t+1) with settle(t); settle(t) is NOT known at decision time, so the label is a research target, not a tradable return.
  B0 training-rate prior | B1 logistic on ln(settle[t-1]/settle[t-2]) | B2 logistic on underlying return (t-2 -> t-1)
  B3 train-fitted rates by signed-moneyness bucket | B4 logistic on B1+B2+signed log-moneyness (only if events/variable >= 10).
  Selection rule (validation only): a model is "selected" if the Bonferroni-adjusted (8 comparisons) train+eval refit
  interval for its paired Brier difference vs B0 lies entirely below 0. Test results are reported, never used to choose.
Usage: python v4_8_feature_research.py [DATA_DIR] [OUT_DIR]
"""
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))
import v4_7_model_dataset as M
import v4_7_source_audit as A

PREFIX = "V4_8_"
SEED = 20260801
N_BOOT = 1000
ALPHA = 0.05
N_COMPARISONS = 8                      # B1..B4 x {CE, PE} on validation
LOGM_EDGES = [-np.inf, -0.10, -0.03, 0.03, 0.10, np.inf]
LOGM_LABELS = ["DEEP_OTM", "OTM", "ATM", "ITM", "DEEP_ITM"]
MISSING = "MISSING"
WINSOR_PCT = (1, 99)
LAMBDA = 1.0                           # L2 on standardized features (intercept unpenalized); fixed, never tuned
SHRINK = 20                            # m-estimate pseudo-count toward the training prior for bucket rates
MIN_EVENTS_PER_VARIABLE = 10
PRIOR_CLIP = (0.01, 0.99)
SPECS = {
    "B1_lag1_settle_logreturn": ("num", ["lr1"]),
    "B2_lag1_underlying_return": ("num", ["ur1"]),
    "B3_signed_moneyness_bucket": ("bucket", ["bucket"]),
    "B4_logit_lr1_ur1_signedlogm": ("num", ["lr1", "ur1", "sm"]),
}
FEATURE_DEFS = [
    ("lr1", "ln(settle[t-1]/settle[t-2])", "close of day t-1", "winsorize 1/99 pct (train), standardize (train)", "NaN -> train mean (0 after standardizing), counted"),
    ("ur1", "underlying[t-1]/underlying[t-2]-1", "close of day t-1", "winsorize 1/99 pct (train), standardize (train)", "NaN -> train mean (0 after standardizing), counted"),
    ("sm", "CE: ln(S[t-1]/K); PE: ln(K/S[t-1])  (>0 = in the money for both)", "close of day t-1", "winsorize 1/99 pct (train), standardize (train)", "NaN -> train mean, counted"),
    ("bucket", "sm cut at -0.10,-0.03,0.03,0.10 -> DEEP_OTM,OTM,ATM,ITM,DEEP_ITM", "close of day t-1", "train-fitted rates, m-estimate shrink=20", "NaN or unseen label -> MISSING -> training prior, counted"),
    ("target_next_up", "settle[t+1] > settle[t], same contract, t+1 = next source trading day", "close of day t+1", "none", "no next consecutive day -> NaN -> ineligible"),
]


class GateError(RuntimeError):
    pass


# ----------------------------------------------------------------------------- gate (D3)
def check_gate(data_dir, n_source_rows=None):
    """Fail closed unless V4.7 audit reports exist, are clean, and belong to the current input files."""
    d = Path(data_dir)
    need = ["Validation_Checks", "Input_Manifest", "Row_Level_Comparison"]
    for n in need:
        if not (d / f"{A.OUT_PREFIX}{n}.csv").exists():
            raise GateError(f"V4.7 audit report missing: {A.OUT_PREFIX}{n}.csv (run the audit first)")
    vc = pd.read_csv(d / f"{A.OUT_PREFIX}Validation_Checks.csv")
    bad = vc[vc["result"] == "REVIEW"]
    if len(bad):
        raise GateError("V4.7 audit has unresolved REVIEW checks:\n" + bad[["scope", "check", "issues"]].to_string(index=False))
    man = pd.read_csv(d / f"{A.OUT_PREFIX}Input_Manifest.csv")
    tracked = man[man["role"].eq("source") | man["format"].isin(["udiff", "legacy"])]
    for r in tracked.itertuples():
        f = d / r.file
        if not f.exists():
            raise GateError(f"audit manifest lists {r.file} but it is not present")
        if A.sha256(f) != r.sha256:
            raise GateError(f"{r.file} changed since the audit ran (hash mismatch): rerun the V4.7 audit")
    listed = set(tracked["file"])
    for f in list(d.glob(A.SOURCE_GLOB)) + list(d.glob("BhavCopy_NSE_FO_*.csv*")):
        if " (" not in f.name and f.name not in listed:
            raise GateError(f"{f.name} is present but was not part of the audit: rerun the V4.7 audit")
    if n_source_rows is not None:
        rl = pd.read_csv(d / f"{A.OUT_PREFIX}Row_Level_Comparison.csv", usecols=["date"])
        if len(rl) != n_source_rows:
            raise GateError(f"audit row-level report has {len(rl)} rows, source has {n_source_rows}: stale audit")
    return len(tracked)


# ----------------------------------------------------------------------------- features (D1, D2)
def signed_log_moneyness(spot, strike, option_type):
    """>0 means in the money for BOTH calls and puts. Raises on unknown option types."""
    spot, strike, ot = np.asarray(spot, float), np.asarray(strike, float), np.asarray(option_type)
    if not np.isin(ot, ["CE", "PE"]).all():
        raise ValueError("option_type must be CE or PE")
    with np.errstate(divide="ignore", invalid="ignore"):
        m = np.log(spot / strike)
    m = np.where(np.isfinite(m), m, np.nan)
    return np.where(ot == "CE", m, -m)


def bucketize(sm):
    sm = np.asarray(sm, float)
    out = np.full(len(sm), MISSING, dtype=object)
    ok = np.isfinite(sm)
    out[ok] = pd.cut(sm[ok], LOGM_EDGES, labels=LOGM_LABELS, right=False).astype(str)
    return out


def add_v48_features(r):
    r = r.copy()
    with np.errstate(divide="ignore", invalid="ignore"):
        lr = np.log1p(r["settle_return_lag1"].to_numpy(float))
    r["lr1"] = np.where(np.isfinite(lr), lr, np.nan)
    r["ur1"] = r["underlying_return_lag1"].astype(float)
    r["sm"] = signed_log_moneyness(r["moneyness_lag1"], 1.0, r["option_type"])   # moneyness_lag1 is already S[t-1]/K
    r["bucket"] = bucketize(r["sm"])
    return r


# ----------------------------------------------------------------------------- train-only transformations
class Standardizer:
    """Winsorize + standardize one numeric column; every parameter comes from the training data only."""

    def fit(self, x):
        x = np.asarray(x, float)
        x = x[np.isfinite(x)]
        self.lo, self.hi = np.percentile(x, WINSOR_PCT)
        c = np.clip(x, self.lo, self.hi)
        self.mu, sd = float(c.mean()), float(c.std())
        self.sd = sd if sd > 0 else 1.0
        return self

    def transform(self, x):
        x = np.asarray(x, float)
        z = (np.clip(x, self.lo, self.hi) - self.mu) / self.sd   # clip(NaN) stays NaN
        self.n_imputed = int((~np.isfinite(z)).sum())
        return np.where(np.isfinite(z), z, 0.0)                    # 0 = training mean


class BucketRates:
    def fit(self, labels, y):
        labels, y = np.asarray(labels, object), np.asarray(y, float)
        self.prior = float(y.mean())
        self.rates = {}
        for k in LOGM_LABELS:
            m = labels == k
            if m.any():
                self.rates[k] = (float(y[m].sum()) + SHRINK * self.prior) / (m.sum() + SHRINK)
        self.counts = {k: int((labels == k).sum()) for k in LOGM_LABELS}
        return self

    def transform(self, labels):
        labels = np.asarray(labels, object)
        p = np.array([self.rates.get(k, self.prior) for k in labels], float)   # unseen or MISSING -> prior
        self.n_fallback = int(sum(k not in self.rates for k in labels))
        return p


def fit_logit(X, y, lam=LAMBDA, iters=60):
    n, k = X.shape
    Z = np.c_[np.ones(n), X]
    w, pen = np.zeros(k + 1), np.r_[0.0, np.full(k, lam)]
    for _ in range(iters):
        p = 1 / (1 + np.exp(-np.clip(Z @ w, -30, 30)))
        g = Z.T @ (p - y) + pen * w
        H = (Z.T * (p * (1 - p))) @ Z + np.diag(pen) + 1e-9 * np.eye(k + 1)
        step = np.linalg.solve(H, g)
        w -= step
        if np.abs(step).max() < 1e-9:
            break
    return w


def fit_model(kind, cols, arrs, y):
    if kind == "num":
        tx = [Standardizer().fit(arrs[c]) for c in cols]
        X = np.column_stack([t.transform(arrs[c]) for t, c in zip(tx, cols)])
        return {"kind": kind, "cols": cols, "tx": tx, "w": fit_logit(X, y)}
    return {"kind": kind, "cols": cols, "tab": BucketRates().fit(arrs["bucket"], y)}


def predict_model(mod, arrs):
    if mod["kind"] == "num":
        X = np.column_stack([t.transform(arrs[c]) for t, c in zip(mod["tx"], mod["cols"])])
        p = 1 / (1 + np.exp(-np.clip(np.c_[np.ones(len(X)), X] @ mod["w"], -30, 30)))
    else:
        p = mod["tab"].transform(arrs["bucket"])
    return np.clip(p, 1e-6, 1 - 1e-6)


def arrays(df):
    return {c: df[c].to_numpy() for c in ("lr1", "ur1", "sm", "bucket")}


def sub(arrs, idx):
    return {k: v[idx] for k, v in arrs.items()}


def prior_prob(y):
    return float(np.clip(np.mean(y), *PRIOR_CLIP))


def b4_justified(y_train):
    events = min(int(np.sum(y_train == 1)), int(np.sum(y_train == 0)))
    nvar = len(SPECS["B4_logit_lr1_ur1_signedlogm"][1])
    return events / nvar >= MIN_EVENTS_PER_VARIABLE, events, nvar


# ----------------------------------------------------------------------------- metrics + paired evaluation
def brier(p, y):
    return float(np.mean((np.asarray(p) - np.asarray(y)) ** 2))


def logloss(p, y):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    y = np.asarray(y, float)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def auc(y, s):
    y = np.asarray(y)
    n1, n0 = y.sum(), len(y) - y.sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    rk = pd.Series(s).rank().to_numpy()
    return float((rk[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def predict_all(tr, te):
    """Fit on tr only, predict te. Every model returns a Series indexed by te.index (paired rows)."""
    y = tr["target_next_up"].to_numpy(float)
    a_tr, a_te = arrays(tr), arrays(te)
    preds = {"B0_training_prior": pd.Series(prior_prob(y), index=te.index)}
    skipped = {}
    for name, (kind, cols) in SPECS.items():
        if name.startswith("B4"):
            ok, ev, nv = b4_justified(y)
            if not ok:
                skipped[name] = f"B4 skipped: {ev} minority-class events / {nv} variables < {MIN_EVENTS_PER_VARIABLE}"
                continue
        preds[name] = pd.Series(predict_model(fit_model(kind, cols, a_tr, y), a_te), index=te.index)
    assert_same_rows(preds)
    return preds, skipped


def assert_same_rows(preds):
    ref = None
    for k, s in preds.items():
        ref = s.index if ref is None else ref
        if not s.index.equals(ref):
            raise AssertionError(f"model {k} evaluated on different rows than the others")


# ----------------------------------------------------------------------------- uncertainty (D4)
def date_groups(dates):
    u, inv = np.unique(dates, return_inverse=True)
    return [np.where(inv == j)[0] for j in range(len(u))], inv


def bootstrap(tr, te, preds, seed, n_boot=None):
    """Paired Brier difference (model - B0) with two interval methods.
    eval_only : resample evaluation DATES; fitted models held fixed.
    refit     : resample training dates AND evaluation dates, refit every model (and the prior) on the resampled
                training rows, score on the resampled evaluation rows.  Captures sampling noise in fitted parameters and in
                the evaluation window. It does NOT capture serial dependence between neighbouring dates, contract
                selection, structural change, data errors, or the use of the same window in earlier experiments."""
    n_boot = N_BOOT if n_boot is None else n_boot
    rng = np.random.default_rng(seed)
    y_te = te["target_next_up"].to_numpy(float)
    y_tr = tr["target_next_up"].to_numpy(float)
    g_te, id_te = date_groups(te["date"].to_numpy())
    g_tr, _ = date_groups(tr["date"].to_numpy())
    se0 = (preds["B0_training_prior"].to_numpy() - y_te) ** 2
    models = [m for m in preds if not m.startswith("B0")]
    out = {m: {"eval_only": np.empty(n_boot), "refit": np.full(n_boot, np.nan)} for m in models}
    n_g = np.array([len(g) for g in g_te])
    s_g = {m: np.array([(((preds[m].to_numpy() - y_te) ** 2) - se0)[g].sum() for g in g_te]) for m in models}
    a_tr, a_te = arrays(tr), arrays(te)
    for b in range(n_boot):
        je = rng.integers(0, len(g_te), len(g_te))
        jt = rng.integers(0, len(g_tr), len(g_tr))
        den = n_g[je].sum()
        for m in models:
            out[m]["eval_only"][b] = s_g[m][je].sum() / den
        idx = np.concatenate([g_tr[i] for i in jt])
        yb = y_tr[idx]
        if yb.min() == yb.max():
            continue
        w = np.bincount(je, minlength=len(g_te))[id_te].astype(float)
        se0b = (prior_prob(yb) - y_te) ** 2
        at = sub(a_tr, idx)
        for m in models:
            kind, cols = SPECS[m]
            pb = predict_model(fit_model(kind, cols, at, yb), a_te)
            out[m]["refit"][b] = np.sum(w * ((pb - y_te) ** 2 - se0b)) / w.sum()
    return out


def interval(x, level):
    lo, hi = np.nanpercentile(x, [100 * (1 - level) / 2, 100 * (1 - (1 - level) / 2)])
    return float(lo), float(hi)


def calibration(y, p, n_min=200, n_bins=5):
    if len(y) < n_min or np.unique(p).size < 2:
        return None
    q = pd.qcut(pd.Series(p), min(n_bins, np.unique(p).size), duplicates="drop")
    d = pd.DataFrame({"y": y, "p": p, "bin": q.astype(str)}).groupby("bin", observed=True).agg(
        n=("y", "size"), mean_pred=("p", "mean"), obs_rate=("y", "mean")).reset_index()
    return d


# ----------------------------------------------------------------------------- orchestration
def run_experiment(d):
    """d: eligible rows with split in (train, validation, test). Returns report frames."""
    comp, ivals, calib, bucket_rows = [], [], [], []
    adj_level = 1 - ALPHA / N_COMPARISONS
    for k, ot in enumerate(sorted(d["option_type"].unique())):
        tr = d[(d["split"] == "train") & (d["option_type"] == ot)]
        if tr["target_next_up"].nunique() < 2:
            raise RuntimeError(f"{ot}: training data has a single class; cannot fit baselines")
        rates = BucketRates().fit(tr["bucket"].to_numpy(), tr["target_next_up"].to_numpy(float))
        for lab in LOGM_LABELS:
            bucket_rows.append({"option_type": ot, "bucket": lab, "train_rows": rates.counts[lab],
                                "rate_shrunk": round(rates.rates.get(lab, rates.prior), 5), "train_prior": round(rates.prior, 5)})
        for j, sp in enumerate(("validation", "test")):
            te = d[(d["split"] == sp) & (d["option_type"] == ot)]
            base = {"option_type": ot, "split": sp, "status": "confirmatory-style (selection split)" if sp == "validation"
                    else "EXPLORATORY: window overlaps data used in earlier experiments; never used to choose"}
            if te.empty:
                comp.append({**base, "model": "(no eligible rows)", "eval_rows": 0, "eval_dates": 0})
                continue
            preds, skipped = predict_all(tr, te)
            y = te["target_next_up"].to_numpy(float)
            b0b, b0l = brier(preds["B0_training_prior"], y), logloss(preds["B0_training_prior"], y)
            boots = bootstrap(tr, te, preds, SEED + 100 * k + 10 * j)
            for name, p in preds.items():
                row = {**base, "model": name, "eval_rows": len(te), "eval_dates": te["date"].nunique(),
                       "train_rows": len(tr), "train_dates": tr["date"].nunique(),
                       "train_up_rate": round(float(tr["target_next_up"].mean()), 4), "eval_up_rate": round(float(y.mean()), 4),
                       "brier": round(brier(p, y), 5), "log_loss": round(logloss(p, y), 5), "auc": round(auc(y, p.to_numpy()), 4),
                       "accuracy_at_0.5": round(float(((p.to_numpy() >= 0.5) == (y == 1)).mean()), 4),
                       "brier_diff_vs_B0": round(brier(p, y) - b0b, 5), "logloss_diff_vs_B0": round(logloss(p, y) - b0l, 5),
                       "selected_on_validation": ""}
                if name != "B0_training_prior":
                    for meth in ("eval_only", "refit"):
                        for lvl, tag in ((1 - ALPHA, f"{int((1 - ALPHA) * 100)}%_unadjusted"),
                                         (adj_level, f"{adj_level * 100:.3f}%_bonferroni_{N_COMPARISONS}")):
                            lo, hi = interval(boots[name][meth], lvl)
                            ivals.append({"option_type": ot, "split": sp, "model": name, "metric": "brier_diff_vs_B0",
                                          "method": meth, "level": tag, "point": row["brier_diff_vs_B0"], "lo": round(lo, 5),
                                          "hi": round(hi, 5), "n_boot": N_BOOT, "valid_replicates": int(np.isfinite(boots[name][meth]).sum()),
                                          "eval_rows": len(te), "eval_dates": te["date"].nunique(), "train_dates": tr["date"].nunique()})
                            if sp == "validation" and meth == "refit" and "bonferroni" in tag:
                                row["selected_on_validation"] = bool(hi < 0)
                comp.append(row)
                c = calibration(y, p.to_numpy())
                if c is not None and name != "B0_training_prior":
                    calib.append(c.assign(option_type=ot, split=sp, model=name))
            for name, why in skipped.items():
                comp.append({**base, "model": name, "eval_rows": len(te), "eval_dates": te["date"].nunique(), "note": why})
    cal = pd.concat(calib, ignore_index=True) if calib else pd.DataFrame(
        columns=["bin", "n", "mean_pred", "obs_rate", "option_type", "split", "model"])
    return pd.DataFrame(comp), pd.DataFrame(ivals), cal, pd.DataFrame(bucket_rows)


def run(data_dir="/content", out_dir=None, verbose=True):
    data_dir, out_dir = Path(data_dir), Path(out_dir or data_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    checks = A.Checks()
    src, _ = A.load_source(str(data_dir), checks)
    n_gate_files = check_gate(data_dir, n_source_rows=len(src))
    checks.add("gate", "V4.7 audit gate (clean, current hashes, row count)", 0, f"{n_gate_files} input files hash-verified")

    r, cal, n_und = M.build_features(src)                      # lags + target on the FULL series, before any filter
    checks.add("v47", "dates with more than one underlying value", n_und)
    rowlevel = pd.read_csv(data_dir / f"{A.OUT_PREFIX}Row_Level_Comparison.csv")
    r = M.apply_rules(r, rowlevel)                             # eligibility AFTER lags
    r, val_start, test_start = M.assign_splits(r, cal)
    M.leakage_checks(r, val_start, test_start, checks)
    r = add_v48_features(r)

    el = r[r["eligible"] & r["split"].isin(["train", "validation", "test"])].copy()
    # V4.8 assertions (fail loudly)
    bad = []
    if (el.loc[el["split"] == "train", "target_date"] >= val_start).any(): bad.append("train label reaches validation")
    if (el.loc[el["split"] == "validation", "target_date"] >= test_start).any(): bad.append("validation label reaches test")
    if (el["target_date"] <= el["date"]).any(): bad.append("target date not after feature date")
    if el[M.FEATURES + ["lr1", "ur1", "sm", "target_next_up"]].isna().any().any(): bad.append("NaN in eligible rows")
    if el["bucket"].eq(MISSING).any(): bad.append("MISSING bucket among eligible rows")
    if bad:
        raise AssertionError("; ".join(bad))
    checks.add("v48", "critical assertions (split labels, target order, NaN, buckets)", 0)

    v47 = data_dir / f"{M.MODEL_PREFIX}Dataset.csv"
    if v47.exists():
        old = pd.read_csv(v47)
        same = set(map(tuple, old.loc[old["eligible"], ["option_type", "strike", "date"]].astype(str).to_numpy())) == \
            set(map(tuple, r.loc[r["eligible"], ["option_type", "strike", "date"]].assign(
                date=lambda x: x["date"].dt.strftime("%Y-%m-%d")).astype(str).to_numpy()))
        checks.add("v47", "eligible row set identical to V4_7_Model_Dataset.csv", 0 if same else 1)
    else:
        checks.add("v47", "eligible row set identical to V4_7_Model_Dataset.csv (file not present; not compared)", 0)

    comp, ivals, calib, brate = run_experiment(el)
    checks.add("v48", "paired evaluation: identical rows for all models", 0, "asserted inside predict_all")

    # ---- reports
    w = A.write_csv
    cols = ["source_file", "symbol", "expiry", "option_type", "strike", "date", "target_date", "cal_idx", "settle_t", "settle_next",
            "lr1", "ur1", "sm", "bucket", "target_next_up", "target_tie", "extreme_return_flag", "official_status", "eligible",
            "exclusion_reason", "split", "split_if_not_purged"]
    ds = r[cols].copy()
    for c in ("expiry", "date", "target_date"):
        ds[c] = ds[c].dt.strftime("%Y-%m-%d")
    w(ds.sort_values(["date", "option_type", "strike"]).round(8).reset_index(drop=True), out_dir / f"{PREFIX}Dataset.csv")
    rules = [x[0] for x in M.RULES]
    elig = pd.DataFrame({"rule": rules, "condition": [x[1] for x in M.RULES],
                         "rows_failing": [int((~r[k]).sum()) for k in rules],
                         "rows_failing_first": [int(r["exclusion_reason"].str.split("|").str[0].eq(k).sum()) for k in rules]})
    elig.loc[len(elig)] = ["ELIGIBLE", "all rules pass", int(r["eligible"].sum()), 0]
    elig.loc[len(elig)] = ["TOTAL", "raw source rows kept in dataset (none deleted)", len(r), 0]
    elig.loc[len(elig)] = ["TIES", "eligible rows where settle[t+1]==settle[t] (counted as not-up)", int(r.loc[r["eligible"], "target_tie"].sum()), 0]
    w(elig, out_dir / f"{PREFIX}Dataset_Eligibility.csv")

    integ = []
    for (sp, ot), g in r[r["split"].isin(["train", "validation", "test", "purged"])].groupby(["split", "option_type"]):
        e = g[g["eligible"]]
        integ.append({"split": sp, "option_type": ot, "rows": len(g), "eligible_rows": len(e),
                      "first_feature_date": g["date"].min().date(), "last_feature_date": g["date"].max().date(),
                      "max_target_date_eligible": e["target_date"].max().date() if len(e) else "",
                      "validation_start": pd.Timestamp(val_start).date(), "test_start": pd.Timestamp(test_start).date(),
                      "train_labels_reaching_validation": int(((e["target_date"] >= val_start).sum()) if sp == "train" else 0),
                      "validation_labels_reaching_test": int(((e["target_date"] >= test_start).sum()) if sp == "validation" else 0)})
    w(pd.DataFrame(integ), out_dir / f"{PREFIX}Split_Integrity.csv")
    w(pd.DataFrame(FEATURE_DEFS, columns=["feature", "definition", "available_at", "train_only_transform", "missing_unseen_policy"]),
      out_dir / f"{PREFIX}Feature_Definitions.csv")
    w(comp, out_dir / f"{PREFIX}Baseline_Comparison.csv")
    w(ivals, out_dir / f"{PREFIX}Uncertainty_Intervals.csv")
    w(calib.round(5), out_dir / f"{PREFIX}Calibration.csv")
    w(brate, out_dir / f"{PREFIX}Moneyness_Bucket_Rates.csv")
    w(checks.frame(), out_dir / f"{PREFIX}Validation_Checks.csv")
    names = sorted(p.name for p in out_dir.glob(PREFIX + "*.csv"))
    zip_path = out_dir / f"{PREFIX}Reports.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for n in names:
            zi = zipfile.ZipInfo(n, date_time=(2026, 1, 1, 0, 0, 0))   # fixed timestamp -> byte-identical reruns
            z.writestr(zi, (out_dir / n).read_bytes())

    if verbose:
        print("=" * 76, "\nV4.8 CONTROLLED FEATURE RESEARCH (paper research; no skill or profit claim)\n" + "=" * 76)
        print(f"Gate OK ({n_gate_files} input files hash-verified). Rows {len(r)} | eligible {int(r['eligible'].sum())} | purged {int((r['split']=='purged').sum())}")
        print(f"Validation starts {pd.Timestamp(val_start).date()} | test starts {pd.Timestamp(test_start).date()} (test = exploratory)")
        show = ["option_type", "split", "model", "eval_rows", "eval_dates", "brier", "brier_diff_vs_B0", "log_loss", "auc", "selected_on_validation"]
        print(comp[[c for c in show if c in comp.columns]].to_string(index=False))
        sel = comp[(comp["split"] == "validation") & (comp["selected_on_validation"] == True)]
        print("\nSelected on validation (Bonferroni refit interval below 0):", "NONE" if sel.empty else sel[["option_type", "model"]].values.tolist())
        print("Intervals: see Uncertainty_Intervals.csv. Results are not confirmatory evidence of skill or profitability.")
        print("Saved", len(names) + 1, "files incl.", zip_path.name, "to", out_dir)
    return {"comparison": comp, "intervals": ivals, "calibration": calib, "dataset": ds, "checks": checks.frame()}


if __name__ == "__main__":
    a = [x for x in sys.argv[1:] if not x.startswith("--")]
    run(a[0] if a else "/content", a[1] if len(a) > 1 else None)


In [ ]:
%%writefile /content/test_v4_8_feature_research.py
"""Automated tests for V4.8 (stdlib unittest). Run: python -m unittest test_v4_8_feature_research -v"""
import shutil
import sys
import tempfile
import unittest
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))
import v4_7_model_dataset as M
import v4_7_source_audit as A
import v4_8_feature_research as V

V.N_BOOT = 60   # keep the suite fast; the production default (1000) is exercised by the real run


def synth_frame(n_dates=14, drop=None, seed=0, low_day=None):
    """Source-like frame as returned by A.load_source: 1 expiry, CE+PE, 3 strikes. drop=(type,strike,day_index)."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2026-01-05", periods=n_dates)
    rows = []
    for ot in ("CE", "PE"):
        for strike in (21000.0, 23000.0, 25000.0):
            px = 100.0
            for i, d in enumerate(dates):
                px = max(px * (1 + rng.normal(0, 0.06)), 3.0)
                if low_day is not None and (ot, strike) == ("CE", 21000.0) and i == low_day:
                    px = 0.5
                if drop and (ot, strike, i) == drop:
                    continue
                rows.append({"source_file": "x.csv", "symbol": "NIFTY", "expiry": pd.Timestamp("2026-09-29"), "option_type": ot,
                             "strike": strike, "date": d, "src_settle": round(px, 2), "src_underlying": 23000 + 40 * i + 15 * (i % 3)})
    return pd.DataFrame(rows), dates


def write_source_csv(path, n_dates=60, seed=3):
    df, _ = synth_frame(n_dates=n_dates, seed=seed)
    out = pd.DataFrame({"Symbol": df.symbol, "Date": df.date.dt.strftime("%d-%b-%Y"), "Expiry": "29-Sep-2026",
                        "Option type": df.option_type, "Strike Price": df.strike, "Close": df.src_settle,
                        "Settle Price": df.src_settle, "Underlying Value": df.src_underlying})
    out.to_csv(path, index=False)


def feats(src):
    r, cal, _ = M.build_features(src)
    return V.add_v48_features(r), cal


class TestMoneyness(unittest.TestCase):
    def test_hand_worked_call_put(self):
        # S=23000, K=24000: call is OTM (S<K), put is ITM. ln(23000/24000) = -0.0425596
        ce = V.signed_log_moneyness([23000], [24000], ["CE"])[0]
        pe = V.signed_log_moneyness([23000], [24000], ["PE"])[0]
        self.assertAlmostEqual(ce, -0.0425596, places=6)
        self.assertAlmostEqual(pe, +0.0425596, places=6)
        self.assertEqual(V.bucketize([ce, pe]).tolist(), ["OTM", "ITM"])
        # S=23000, K=15000: call deep ITM, put deep OTM. ln(23000/15000) = 0.427444
        self.assertEqual(V.bucketize(V.signed_log_moneyness([23000, 23000], [15000, 15000], ["CE", "PE"])).tolist(),
                         ["DEEP_ITM", "DEEP_OTM"])
        self.assertEqual(V.bucketize([0.0])[0], "ATM")

    def test_bucket_edges_are_left_closed(self):
        self.assertEqual(V.bucketize([-0.10, -0.03, 0.03, 0.10]).tolist(), ["OTM", "ATM", "ITM", "DEEP_ITM"])

    def test_same_ratio_opposite_meaning_by_type(self):
        r, _ = feats(synth_frame()[0])
        row = r.dropna(subset=["moneyness_lag1"])
        ce, pe = row[row.option_type == "CE"], row[row.option_type == "PE"]
        np.testing.assert_allclose(ce["sm"], np.log(ce["moneyness_lag1"]))
        np.testing.assert_allclose(pe["sm"], -np.log(pe["moneyness_lag1"]))

    def test_rejects_unknown_type_and_marks_nan(self):
        with self.assertRaises(ValueError):
            V.signed_log_moneyness([1], [1], ["XX"])
        self.assertEqual(V.bucketize([np.nan, np.inf])[0], V.MISSING)

    def test_lr1_is_log_of_price_ratio(self):
        src, dates = synth_frame()
        r, _ = feats(src)
        c = src[(src.option_type == "CE") & (src.strike == 23000.0)].set_index("date").src_settle
        row = r[(r.option_type == "CE") & (r.strike == 23000.0) & (r.date == dates[6])].iloc[0]
        self.assertAlmostEqual(row.lr1, np.log(c[dates[5]] / c[dates[4]]), places=12)


class TestNoLookahead(unittest.TestCase):
    def test_future_prices_do_not_change_past_features_or_labels(self):
        src, dates = synth_frame()
        T = dates[8]
        other = src.copy()
        fut = other["date"] > T
        other.loc[fut, "src_settle"] *= 3.7
        other.loc[fut, "src_underlying"] += 900
        a, b = feats(src)[0], feats(other)[0]
        cols = M.FEATURES + ["lr1", "ur1", "sm"]
        past = a["date"] <= T
        pd.testing.assert_frame_equal(a.loc[past, cols], b.loc[past, cols])
        lab = a["date"] < T                         # label at t uses t+1 <= T
        pd.testing.assert_series_equal(a.loc[lab, "target_next_up"], b.loc[lab, "target_next_up"])

    def test_features_at_t_ignore_price_at_t(self):
        src, dates = synth_frame()
        T = dates[8]
        other = src.copy()
        m = other["date"] == T
        other.loc[m, "src_settle"] *= 5
        other.loc[m, "src_underlying"] += 700
        a, b = feats(src)[0], feats(other)[0]
        cols = M.FEATURES + ["lr1", "ur1", "sm"]
        same = a["date"] <= T
        pd.testing.assert_frame_equal(a.loc[same, cols], b.loc[same, cols])
        self.assertFalse(a.loc[a.date == dates[9], cols].equals(b.loc[b.date == dates[9], cols]))  # it enters the NEXT day only


class TestGapsAndLagOrder(unittest.TestCase):
    def test_missing_date_breaks_target_and_lags(self):
        src, dates = synth_frame(drop=("CE", 21000.0, 6))
        r, _ = feats(src)
        c = r[(r.option_type == "CE") & (r.strike == 21000.0)].set_index("date")
        self.assertTrue(pd.isna(c.loc[dates[5], "target_next_up"]))
        self.assertTrue(pd.isna(c.loc[dates[7], "settle_return_lag1"]) and pd.isna(c.loc[dates[8], "settle_return_lag1"]))
        self.assertFalse(pd.isna(c.loc[dates[9], "settle_return_lag1"]))
        self.assertTrue(pd.isna(c.loc[dates[7], "lr1"]))

    def test_lags_are_computed_before_eligibility_filters(self):
        src, dates = synth_frame(low_day=6)          # CE 21000 settles at 0.5 (< floor) on day 6
        r, _ = feats(src)
        rl = r[["date", "option_type", "strike", "expiry"]].assign(audit_status=A.ST_NO_DATE)
        for c in ("date", "expiry"):
            rl[c] = rl[c].dt.strftime("%Y-%m-%d")
        r = M.apply_rules(r, rl)
        c = r[(r.option_type == "CE") & (r.strike == 21000.0)].set_index("date")
        # day 7's lag uses the real day-6 price although day 6 is filtered out by the floor
        s = src[(src.option_type == "CE") & (src.strike == 21000.0)].set_index("date").src_settle
        self.assertAlmostEqual(c.loc[dates[8], "settle_return_lag1"], s[dates[7]] / s[dates[6]] - 1, places=12)
        self.assertFalse(c.loc[dates[6], "R3_price_floor"])
        self.assertFalse(c.loc[dates[8], "R3_price_floor"])        # t-2 is the low price
        self.assertTrue(c.loc[dates[8], "R1_features"])             # feature exists; only the filter removes the row


class TestTrainOnlyAndPolicies(unittest.TestCase):
    def test_standardizer_params_come_from_train_only(self):
        tr, va = np.arange(100.0), np.array([1e6, -1e6, np.nan])
        s = V.Standardizer().fit(tr)
        before = (s.lo, s.hi, s.mu, s.sd)
        z = s.transform(va)
        self.assertEqual(before, (s.lo, s.hi, s.mu, s.sd))               # transform does not refit
        both = V.Standardizer().fit(np.r_[tr, va[:2]])
        self.assertNotEqual(before, (both.lo, both.hi, both.mu, both.sd))  # contamination would show
        self.assertTrue(np.isfinite(z).all())
        self.assertEqual(s.n_imputed, 1)
        self.assertEqual(z[2], 0.0)                                       # NaN -> training mean

    def test_bucket_rates_train_only_and_unseen_policy(self):
        lab = np.array(["OTM"] * 40 + ["ATM"] * 40, dtype=object)
        y = np.r_[np.ones(40), np.zeros(40)]
        t = V.BucketRates().fit(lab, y)
        self.assertAlmostEqual(t.prior, 0.5)
        self.assertAlmostEqual(t.rates["OTM"], (40 + V.SHRINK * 0.5) / (40 + V.SHRINK))
        p = t.transform(np.array(["OTM", "DEEP_ITM", V.MISSING, "never-seen"], dtype=object))
        self.assertEqual(p[1], t.prior)
        self.assertEqual(p[2], t.prior)
        self.assertEqual(p[3], t.prior)
        self.assertEqual(t.n_fallback, 3)
        self.assertNotIn("DEEP_ITM", t.rates)

    def test_predictions_never_read_evaluation_labels(self):
        src, _ = synth_frame(n_dates=40)
        r, cal = feats(src)
        r = r.dropna(subset=["lr1", "ur1", "sm", "target_next_up"])
        tr, te = r.iloc[: len(r) // 2], r.iloc[len(r) // 2:].copy()
        p1, _ = V.predict_all(tr, te)
        te2 = te.copy()
        te2["target_next_up"] = 1 - te2["target_next_up"]
        p2, _ = V.predict_all(tr, te2)
        for k in p1:
            pd.testing.assert_series_equal(p1[k], p2[k])
        tr2 = tr.copy()
        tr2["target_next_up"] = 1 - tr2["target_next_up"]
        p3, _ = V.predict_all(tr2, te)
        self.assertFalse(p1["B3_signed_moneyness_bucket"].equals(p3["B3_signed_moneyness_bucket"]))

    def test_missing_and_extreme_inputs_give_finite_predictions(self):
        src, _ = synth_frame(n_dates=40)
        r, _ = feats(src)
        r = r.dropna(subset=["lr1", "ur1", "sm", "target_next_up"])
        te = r.iloc[-20:].copy()
        te.iloc[0, te.columns.get_loc("lr1")] = np.nan
        te.iloc[1, te.columns.get_loc("ur1")] = 1e9
        te.iloc[2, te.columns.get_loc("bucket")] = "BRAND_NEW"
        p, _ = V.predict_all(r.iloc[:-20], te)
        for s in p.values():
            self.assertTrue(np.isfinite(s.to_numpy()).all())
            self.assertTrue(((s > 0) & (s < 1)).all())

    def test_b4_only_when_events_justify(self):
        ok, ev, nv = V.b4_justified(np.r_[np.ones(40), np.zeros(60)])
        self.assertTrue(ok)
        ok2, _, _ = V.b4_justified(np.r_[np.ones(20), np.zeros(200)])
        self.assertFalse(ok2)


class TestPairedAndMetrics(unittest.TestCase):
    def test_assert_same_rows_rejects_mismatch(self):
        a = pd.Series([0.5, 0.5], index=[1, 2])
        with self.assertRaises(AssertionError):
            V.assert_same_rows({"B0": a, "B1": pd.Series([0.4, 0.6], index=[1, 3])})

    def test_all_models_share_row_index(self):
        src, _ = synth_frame(n_dates=40)
        r, _ = feats(src)
        r = r.dropna(subset=["lr1", "ur1", "sm", "target_next_up"])
        p, _ = V.predict_all(r.iloc[:60], r.iloc[60:])
        idx = [s.index.tolist() for s in p.values()]
        self.assertTrue(all(i == idx[0] for i in idx))
        self.assertEqual(len(p), 5)

    def test_hand_metrics(self):
        self.assertAlmostEqual(V.brier([0.5, 0.5], [1, 0]), 0.25)
        self.assertAlmostEqual(V.logloss([0.5, 0.5], [1, 0]), np.log(2))
        self.assertAlmostEqual(V.auc([0, 1], [0.1, 0.9]), 1.0)
        self.assertAlmostEqual(V.auc([0, 1, 0, 1], [0.5, 0.5, 0.5, 0.5]), 0.5)
        self.assertTrue(np.isnan(V.auc([1, 1], [0.2, 0.3])))

    def test_logit_intercept_matches_log_odds(self):
        y = np.r_[np.ones(30), np.zeros(70)]
        w = V.fit_logit(np.zeros((100, 1)), y)
        self.assertAlmostEqual(w[0], np.log(0.3 / 0.7), places=5)

    def test_bootstrap_identical_model_has_zero_difference_and_is_seeded(self):
        src, _ = synth_frame(n_dates=40)
        r, _ = feats(src)
        r = r.dropna(subset=["lr1", "ur1", "sm", "target_next_up"])
        tr, te = r.iloc[:80], r.iloc[80:]
        p, _ = V.predict_all(tr, te)
        same = {"B0_training_prior": p["B0_training_prior"]}
        out = V.bootstrap(tr, te, p, seed=5, n_boot=30)
        again = V.bootstrap(tr, te, p, seed=5, n_boot=30)
        for m in out:
            np.testing.assert_array_equal(out[m]["eval_only"], again[m]["eval_only"])
            np.testing.assert_array_equal(out[m]["refit"], again[m]["refit"])
            self.assertTrue(np.isfinite(out[m]["refit"]).sum() > 20)
        p_dup = dict(p)
        p_dup["B1_lag1_settle_logreturn"] = p["B0_training_prior"]
        z = V.bootstrap(tr, te, {k: p_dup[k] for k in ("B0_training_prior", "B1_lag1_settle_logreturn")}, seed=5, n_boot=30)
        np.testing.assert_allclose(z["B1_lag1_settle_logreturn"]["eval_only"], 0.0, atol=1e-12)


class TestPipelineEndToEnd(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.root = Path(tempfile.mkdtemp(prefix="v48test_"))
        cls.data = cls.root / "data"
        cls.data.mkdir()
        write_source_csv(cls.data / "OPTIDX_NIFTY_CE_01-Jan-2026_TO_31-Mar-2026.csv")
        A.run_audit(cls.data, verbose=False)
        cls.out = cls.root / "out1"
        cls.res = V.run(cls.data, cls.out, verbose=False)
        cls.ds = pd.read_csv(cls.out / "V4_8_Dataset.csv", parse_dates=["date", "target_date"])

    @classmethod
    def tearDownClass(cls):
        shutil.rmtree(cls.root, ignore_errors=True)

    def test_no_label_crosses_a_split_boundary(self):
        ds = self.ds
        e = ds[ds.eligible]
        val0 = ds.loc[ds.split == "validation", "date"].min()
        te0 = ds.loc[ds.split == "test", "date"].min()
        self.assertTrue((e.loc[e.split == "train", "target_date"] < val0).all())
        self.assertTrue((e.loc[e.split == "validation", "target_date"] < te0).all())
        self.assertTrue((e["target_date"] > e["date"]).all())

    def test_purge_at_every_boundary(self):
        ds = self.ds
        val0 = ds.loc[ds.split == "validation", "cal_idx"].min()
        te0 = ds.loc[ds.split == "test", "cal_idx"].min()
        for b, origin in ((val0, "train"), (te0, "validation")):
            at = ds[ds.cal_idx == b - 1]
            self.assertGreater(len(at), 0)
            self.assertTrue((at.split == "purged").all())
            self.assertTrue((at.split_if_not_purged == origin).all())
        kept = ds[ds.split.isin(["train", "validation"])]
        nxt = np.where(kept.cal_idx < val0, val0, te0)            # first date of the following split
        self.assertTrue((kept.cal_idx + 1 < nxt).all())

    def test_all_raw_rows_kept(self):
        self.assertEqual(len(self.ds), 360)                       # 2 types x 3 strikes x 60 dates, none deleted
        ex = self.ds["exclusion_reason"].fillna("")
        self.assertTrue((ex[self.ds.eligible] == "").all() and (ex[~self.ds.eligible] != "").all())

    def test_reports_exist_and_checks_pass(self):
        for n in ("Dataset", "Dataset_Eligibility", "Split_Integrity", "Feature_Definitions", "Baseline_Comparison",
                  "Uncertainty_Intervals", "Calibration", "Moneyness_Bucket_Rates", "Validation_Checks"):
            self.assertTrue((self.out / f"V4_8_{n}.csv").exists(), n)
        self.assertTrue((self.out / "V4_8_Reports.zip").exists())
        self.assertTrue((self.res["checks"]["issues"] == 0).all())

    def test_comparison_pairs_rows_and_labels_test_exploratory(self):
        c = self.res["comparison"]
        c = c[c.eval_rows > 0]
        for (ot, sp), g in c.groupby(["option_type", "split"]):
            self.assertEqual(g["eval_rows"].nunique(), 1)
            self.assertIn("B0_training_prior", set(g.model))
        self.assertTrue(c[c.split == "test"]["status"].str.startswith("EXPLORATORY").all())
        self.assertTrue((c[c.split == "test"]["selected_on_validation"] == "").all())

    def test_deterministic_rerun_byte_identical(self):
        out2 = self.root / "out2"
        V.run(self.data, out2, verbose=False)
        for p in sorted(self.out.glob("V4_8_*")):
            self.assertEqual(A.sha256(p), A.sha256(out2 / p.name), p.name)

    def _fresh(self, name):
        d = self.root / name
        shutil.copytree(self.data, d)
        return d

    def test_gate_fails_closed(self):
        d = self._fresh("g1")
        (d / "V4_7_Audit_Input_Manifest.csv").unlink()
        with self.assertRaises(V.GateError):
            V.run(d, self.root / "o_g1", verbose=False)
        d = self._fresh("g2")                                    # audit has unresolved REVIEW item
        vc = pd.read_csv(d / "V4_7_Audit_Validation_Checks.csv")
        vc.loc[0, ["issues", "result"]] = [3, "REVIEW"]
        vc.to_csv(d / "V4_7_Audit_Validation_Checks.csv", index=False)
        with self.assertRaises(V.GateError):
            V.run(d, self.root / "o_g2", verbose=False)
        d = self._fresh("g3")                                    # source changed after the audit
        f = next(d.glob("OPTIDX_NIFTY_*.csv"))
        f.write_text(f.read_text().replace("23000.0", "23001.0", 1))
        with self.assertRaises(V.GateError):
            V.run(d, self.root / "o_g3", verbose=False)
        d = self._fresh("g4")                                    # stale row-level report
        rl = pd.read_csv(d / "V4_7_Audit_Row_Level_Comparison.csv")
        rl.iloc[:-5].to_csv(d / "V4_7_Audit_Row_Level_Comparison.csv", index=False)
        with self.assertRaises(V.GateError):
            V.run(d, self.root / "o_g4", verbose=False)
        d = self._fresh("g5")                                    # new official file the audit never saw
        (d / "BhavCopy_NSE_FO_0_0_0_20260110_F_0000.csv").write_text("TradDt,FinInstrmTp,TckrSymb\n")
        with self.assertRaises(V.GateError):
            V.run(d, self.root / "o_g5", verbose=False)
        self.assertFalse((self.root / "o_g1" / "V4_8_Reports.zip").exists())   # nothing written when the gate fails

    def test_does_not_modify_inputs(self):
        before = {p.name: A.sha256(p) for p in self.data.iterdir()}
        V.run(self.data, self.root / "out3", verbose=False)
        self.assertEqual(before, {p.name: A.sha256(p) for p in self.data.iterdir()})


if __name__ == "__main__":
    unittest.main(verbosity=2)


In [ ]:
#@title ▶ RUN V4.7 AUDIT (gate prerequisite)
DATA_DIR = "/content"
import sys, importlib
sys.path.insert(0, DATA_DIR)
import v4_7_source_audit as A
importlib.reload(A)
audit = A.run_audit(DATA_DIR, DATA_DIR)
bad = audit["checks"][audit["checks"]["result"] == "REVIEW"]
assert bad.empty, "V4.7 audit has unresolved REVIEW checks - stop and inspect:\n" + bad.to_string()

In [ ]:
#@title ▶ RUN TESTS (must pass before the experiment)
import subprocess, sys
res = subprocess.run([sys.executable, "-m", "unittest", "test_v4_8_feature_research", "-v"],
                     cwd="/content", capture_output=True, text=True)
print(res.stderr[-6000:])
assert res.returncode == 0, "V4.8 tests FAILED - do not use any results"

In [ ]:
#@title ▶ RUN V4.8 PIPELINE (gate -> dataset -> baselines -> intervals -> zip)
import importlib, os
import v4_8_feature_research as V
importlib.reload(V)
results = V.run(DATA_DIR, DATA_DIR)
zip_path = os.path.join(DATA_DIR, "V4_8_Reports.zip")
assert os.path.exists(zip_path), "report ZIP missing"
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass